overall

In [0]:
# This cell is for setting up the environment and tools for data auditing in Databricks.
# 1. Install PySpark (not needed in Databricks, but shown for completeness)
# !pip install pyspark -q

# 2. Import tools (not needed in Databricks, as SparkSession and functions are available)
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, min, max, avg, sum

# 3. Initialize the Session (not needed in Databricks, as 'spark' is already available)
# spark = SparkSession.builder.master("local[*]").appName("DataAudit").getOrCreate()

Loading All Files

In [0]:
# Upload CSV files to the volume path in the bronze schema of captone_catalog catalog
import shutil

volume_path = "/Volumes/capstone_catalog/bronze/bronze_volume/"

shutil.copy("/Workspace/capstone_folder/sales_transactions.csv", volume_path + "sales_transactions.csv")
shutil.copy("/Workspace/capstone_folder/product_master.csv", volume_path + "product_master.csv")
shutil.copy("/Workspace/capstone_folder/store_master.csv", volume_path + "store_master.csv")
shutil.copy("/Workspace/capstone_folder/customer_data.csv", volume_path + "customer_data.csv")
shutil.copy("/Workspace/capstone_folder/inventory_data.csv", volume_path + "inventory_data.csv")
shutil.copy("/Workspace/capstone_folder/clickstream_events.csv", volume_path + "clickstream_events.csv")

sales_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(volume_path + "sales_transactions.csv")
prod_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(volume_path + "product_master.csv")
store_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(volume_path + "store_master.csv")
cust_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(volume_path + "customer_data.csv")
inv_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(volume_path + "inventory_data.csv")
click_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(volume_path + "clickstream_events.csv")

dfs = {
    "Sales": sales_df,
    "Products": prod_df,
    "Stores": store_df,
    "Customers": cust_df,
    "Inventory": inv_df,
    "Clickstream": click_df
}

In [0]:
# Create a list of your dataframes
dfs = {
    "Sales": sales_df,
    "Products": prod_df,
    "Stores": store_df,
    "Customers": cust_df,
    "Inventory": inv_df,
    "Clickstream": click_df
}

for name, df in dfs.items():
    print(f"--- Data Quality Audit: {name} ---")

    # 1. Count Total Rows
    print(f"Total Rows: {df.count()}")

    # 2. Check for Nulls in every column
    from pyspark.sql.functions import count, when, isnan, col
    from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType, ShortType, ByteType

    missing_value_expressions = []
    for c in df.columns:
        # Check if the column is of a numeric type that can have NaN (DoubleType or FloatType)
        if isinstance(df.schema[c].dataType, (DoubleType, FloatType)):
            missing_value_expressions.append(count(when(isnan(col(c)) | col(c).isNull(), c)).alias(c))
        else:
            missing_value_expressions.append(count(when(col(c).isNull(), c)).alias(c))
    df.select(missing_value_expressions).show()

    # 3. Check for Duplicates
    duplicate_count = df.count() - df.dropDuplicates().count()
    print(f"Duplicate Rows: {duplicate_count}")
    print("\n")

This code goes through every file and checks for:

Row Count: How much data do we have?

Null Values: Are there "holes" in our data?

Duplicates: Are we counting the same thing twice?

In [0]:
from pyspark.sql.functions import col, to_date, lit
from datetime import datetime
from pyspark.sql.types import DateType, TimestampType

print("--- Checking for Dates Greater Than Present Date ---")

# Get the current date and time
present_datetime = datetime.now()
present_date = present_datetime.date()

print(f"Comparing dates against: {present_date}")

dfs_to_check = {
    "Sales": (sales_df, ["order_date", "ingestion_timestamp"]),
    "Inventory": (inv_df, ["last_updated"]),
    "Clickstream": (click_df, ["event_timestamp"])
}

found_future_dates_overall = False

for df_name, (df, date_cols) in dfs_to_check.items():
    print(f"\n--- Checking DataFrame: {df_name} ---")
    found_future_dates_in_df = False

    for col_name in date_cols:
        # Ensure the column is a date or timestamp type before comparison
        # Convert to_date() if it's a timestamp for date-only comparison, or directly use if already date
        if col_name == "event_timestamp" and df_name == "Clickstream":
            # For clickstream, event_timestamp might be string, convert to timestamp first
            date_column = to_date(col(col_name))
        else:
            # For other DataFrames, assume schema inferencing has made them DateType
            date_column = col(col_name)

        # Filter for dates strictly greater than the present date
        future_records = df.filter(date_column > lit(present_date))

        if future_records.count() > 0:
            found_future_dates_overall = True
            found_future_dates_in_df = True
            print(f"⚠️ Found records in column '{col_name}' with dates greater than {present_date}:")
            future_records.select(col_name).distinct().show(truncate=False)

    if not found_future_dates_in_df:
        print(f"✅ No records with dates greater than {present_date} found in {df_name}.")

if not found_future_dates_overall:
    print("\nOverall: ✅ No records with dates greater than present date found across all checked DataFrames.")
else:
    print("\nOverall: ⚠️ Some records with dates greater than present date were found.")


### Outlier Detection in `quantity` using IQR (Original Raw Data)

This code identifies outliers in the `quantity` column of the *original* `sales_df` DataFrame using the Interquartile Range (IQR) method. Outliers are defined as values that fall below `Q1 - 1.5 * IQR` or above `Q3 + 1.5 * IQR`.

In [0]:
from pyspark.sql.functions import col

print("--- Checking for Outliers in 'quantity' column of ORIGINAL sales_df ---")

# Calculate Q1 and Q3 for 'quantity' in the original sales_df
quantiles_original = sales_df.approxQuantile("quantity", [0.25, 0.75], 0.05)
Q1_original = quantiles_original[0]
Q3_original = quantiles_original[1]
IQR_original = Q3_original - Q1_original

# Define outlier bounds
lower_bound_original = Q1_original - 1.5 * IQR_original
upper_bound_original = Q3_original + 1.5 * IQR_original

print(f"Q1 (original data): {Q1_original}")
print(f"Q3 (original data): {Q3_original}")
print(f"IQR (original data): {IQR_original}")
print(f"Lower Bound (original data): {lower_bound_original}")
print(f"Upper Bound (original data): {upper_bound_original}")

# Identify outliers
outliers_original_df = sales_df.filter(
    (col("quantity") < lower_bound_original) | (col("quantity") > upper_bound_original)
)

outliers_original_count = outliers_original_df.count()
print(f"\nTotal outliers found in 'quantity' column of ORIGINAL sales_df: {outliers_original_count}")

if outliers_original_count > 0:
    print("\nSample of records with outlier 'quantity' values in ORIGINAL sales_df:")
    outliers_original_df.select("transaction_id", "product_id", "quantity", "unit_price", "total_amount").show(20, truncate=False)
else:
    print("✅ No outliers found in the 'quantity' column of ORIGINAL sales_df based on the IQR method.")

checking for discount > 100%

### Referential Integrity Check: Orphan IDs in Sales Data (Original Raw Data)

This check identifies if any `product_id`, `store_id`, or `customer_id` in the *original* `sales_df` DataFrame are missing from their corresponding master dataframes (`prod_df`, `store_df`, `cust_df`).

A missing ID indicates an inconsistency, as a transaction would refer to a non-existent product, store, or customer.

In [0]:
print("--- Checking for Orphan IDs in ORIGINAL sales_df ---")

# Check for Sales with missing Product IDs in prod_df
orphan_products_count = sales_df.join(
    prod_df.select("product_id"), "product_id", "left_anti"
).count()
print(f"Sales records with unknown product_ids (not in prod_df): {orphan_products_count}")

# Check for Sales with missing Store IDs in store_df
orphan_stores_count = sales_df.join(
    store_df.select("store_id"), "store_id", "left_anti"
).count()
print(f"Sales records with unknown store_ids (not in store_df): {orphan_stores_count}")

# Check for Sales with missing Customer IDs in cust_df
orphan_customers_count = sales_df.join(
    cust_df.select("customer_id"), "customer_id", "left_anti"
).count()
print(f"Sales records with unknown customer_ids (not in cust_df): {orphan_customers_count}")

if orphan_products_count == 0 and orphan_stores_count == 0 and orphan_customers_count == 0:
    print("✅ All product_ids, store_ids, and customer_ids in ORIGINAL sales_df have matching entries in their respective master files.")
else:
    print("⚠️ Inconsistencies found in product_id, store_id, or customer_id referential integrity in ORIGINAL sales_df.")

### General City/State Mapping Logic Check in `store_df`

This code identifies inconsistencies in the `store_df` where a single city is mapped to more than one unique state, indicating potential data quality issues.

In [0]:
from pyspark.sql.functions import countDistinct, col

print("--- Checking General City/State Mapping in store_df ---")

# Group by city and count distinct states associated with each city
conflicting_city_state_mappings = store_df.groupBy("city") \
                                           .agg(countDistinct("state").alias("distinct_states")) \
                                           .filter("distinct_states > 1")

inconsistency_count = conflicting_city_state_mappings.count()

if inconsistency_count > 0:
    print(f"⚠️ Found {inconsistency_count} cities mapped to more than one state:")
    conflicting_city_state_mappings.show(truncate=False)
    print("\n--- Sample of records for these inconsistent cities ---")
    # Display actual records for these conflicting cities for further inspection
    store_df.join(conflicting_city_state_mappings, "city").orderBy("city").show(truncate=False)
else:
    print("✅ No cities found that are mapped to more than one state.")

### Cleaning: Standardizing City-State Mappings in `cleaned_store_master`

This code identifies the most frequent state for each city and then updates all records for that city in `cleaned_store_master` to use this canonical state, resolving inconsistencies where a single city was previously mapped to multiple states.

### Store Type Standardization Check in `store_df`

This code verifies that the `store_type` column in `store_df` contains only the allowed standardized values: 'Mall', 'Standalone', or 'Warehouse'. It will display any non-standard entries found.

In [0]:
from pyspark.sql.functions import col

print("\n--- Checking Store Type Standardization in store_df ---")

allowed_store_types = ["Mall", "Standalone", "Warehouse"]

non_standard_store_types = store_df.filter(~col("store_type").isin(allowed_store_types)).distinct()

non_standard_count = non_standard_store_types.count()

if non_standard_count > 0:
    print(f"⚠️ Found {non_standard_count} records with non-standard 'store_type' values:")
    non_standard_store_types.select("store_type").show(truncate=False)
else:
    print("✅ All 'store_type' values are standardized ('Mall', 'Standalone', or 'Warehouse').")

### Checking for Consistency in 'brand' Column (Raw Data)

This code identifies and displays records in the `prod_df` DataFrame where the `brand` column contains common inconsistent or placeholder values like 'None', 'N/A', or 'Unknown'.

In [0]:
from pyspark.sql.functions import col, upper

print("--- Checking 'brand' column for consistency issues in prod_df (Raw Data) ---")

# Define the list of inconsistent strings to check for
inconsistent_strings = ["None", "N/A", "Unknown", "none", "n/a", "unknown"]

# Filter the DataFrame for records where 'brand' matches any of the inconsistent strings
inconsistent_brands_df = prod_df.filter(upper(col("brand")).isin([s.upper() for s in inconsistent_strings]))

# Count and display the inconsistent records
inconsistent_brands_count = inconsistent_brands_df.count()

if inconsistent_brands_count > 0:
    print(f"⚠️ Found {inconsistent_brands_count} records with inconsistent 'brand' values:")
    inconsistent_brands_df.select("product_id", "brand").show(truncate=False)
else:
    print("✅ No inconsistent 'brand' values ('None', 'N/A', 'Unknown') found in prod_df.")

print("\n--- Displaying all distinct brand values for visual inspection ---")
prod_df.select("brand").distinct().show(truncate=False)

### Checking for Spelling Errors in 'category' Column (Raw Data)

This code extracts and displays all unique values from the `category` column in the `prod_df` DataFrame. This allows for a visual inspection to identify any potential spelling errors or inconsistencies in the raw category data.

In [0]:
print("--- Distinct 'category' values in prod_df (Raw Data) ---")
prod_df.select("category").distinct().show(truncate=False)

In [0]:
from pyspark.sql.functions import col

print("--- Checking for Discounts Greater Than 100% (i.e., discount > 1.0) in ORIGINAL sales_df ---")

discount_anomaly_df = sales_df.filter(col("discount") > 1.0)

anomaly_count = discount_anomaly_df.count()

if anomaly_count > 0:
    print(f"⚠️ Found {anomaly_count} records in ORIGINAL sales_df with discount values greater than 1.0:")
    discount_anomaly_df.select("transaction_id", "product_id", "quantity", "unit_price", "discount", "total_amount").show(20, truncate=False)
else:
    print("✅ No records found in ORIGINAL sales_df with discount values greater than 1.0.")

In [0]:
from pyspark.sql.types import DoubleType, FloatType

def run_basic_audit(df_dict):
    for name, df in df_dict.items():
        print(f"\n{'='*20} AUDITING: {name} {'='*20}")

        # Check 1: Total Volume
        total_rows = df.count()
        print(f"Total Records: {total_rows}")

        # Check 2: Missing Values (Nulls)
        # This scans every column and counts how many empty spots exist
        print("Missing Values per Column:")
        missing_value_expressions = []
        for c in df.columns:
            # Check if the column is of a numeric type (DoubleType or FloatType)
            if isinstance(df.schema[c].dataType, (DoubleType, FloatType)):
                missing_value_expressions.append(count(when(isnan(col(c)) | col(c).isNull(), c)).alias(c))
            else:
                missing_value_expressions.append(count(when(col(c).isNull(), c)).alias(c))
        df.select(missing_value_expressions).show()

        # Check 3: Duplicates
        unique_rows = df.dropDuplicates().count()
        if total_rows > unique_rows:
            print(f"⚠️ WARNING: Found {total_rows - unique_rows} duplicate rows!")
        else:
            print("✅ No duplicate rows found.")

run_basic_audit(dfs)

In [0]:
from pyspark.sql.types import IntegerType, DoubleType, FloatType, LongType, ShortType, ByteType

print("--- Checking for Negative Values in Numeric Columns ---")

for name, df in dfs.items():
    print(f"\n{'='*20} Checking DataFrame: {name} {'='*20}")
    found_negative = False
    for col_name in df.columns:
        # Check if the column is a numeric type
        if isinstance(df.schema[col_name].dataType, (IntegerType, DoubleType, FloatType, LongType, ShortType, ByteType)):
            negative_values_df = df.filter(col(col_name) < 0)
            if negative_values_df.count() > 0:
                found_negative = True
                print(f"\n⚠️ Negative values found in column '{col_name}':")
                negative_values_df.show(truncate=False)

    if not found_negative:
        print(f"✅ No negative values found in any numeric column of {name}.")

### Identifying Columns with Duplicate Values (Re-attempt)

This code will iterate through each DataFrame (`sales_df`, `prod_df`, etc.) and then through each column within those DataFrames. For every column, it calculates if the number of distinct values is less than the total number of rows, which indicates the presence of duplicate values within that specific column. This directly addresses your request to find columns that contain duplicate values.

In [0]:
from pyspark.sql.functions import countDistinct

print("\n--- Checking for Duplicate Values in Columns ---")

for name, df in dfs.items():
    print(f"\n{'='*20} Checking DataFrame: {name} {'='*20}")
    for col_name in df.columns:
        total_rows = df.count()
        distinct_values = df.select(countDistinct(col_name)).collect()[0][0]

        if total_rows > 0 and distinct_values < total_rows:
            print(f"⚠️ Column '{col_name}' has duplicate values ({distinct_values} distinct out of {total_rows} total rows).")
        else:
            print(f"✅ Column '{col_name}' has no duplicate values (all {total_rows} values are distinct or dataframe is empty).")


### Customers with conflicting age details

This code identifies `customer_id`s that have more than one unique age associated with them, indicating potential data inconsistencies where a single customer is recorded with different age details.

In [0]:
from pyspark.sql.functions import countDistinct, col

conflicting_age_customers = cust_df.groupBy("customer_id") \
                                   .agg(countDistinct("age").alias("distinct_ages")) \
                                   .filter("distinct_ages > 1")

print("Customers with conflicting age details (same ID, different ages):")
if conflicting_age_customers.count() > 0:
    conflicting_age_customers.show()
    print("Full details for customers with conflicting age information:")
    cust_df.join(conflicting_age_customers, "customer_id").orderBy("customer_id").show(truncate=False)
else:
    print("No customers found with conflicting age details.")

### Customers with conflicting details in any column

This code generalizes the check to find `customer_id`s that have more than one unique value for *any* descriptive column (excluding `customer_id` itself). This is a comprehensive way to detect inconsistencies in customer profiles.

In [0]:
from pyspark.sql.functions import countDistinct, col

print("\n--- Checking for Conflicting Details Across All Columns (per Customer ID) ---")

# Get all columns except 'customer_id'
other_columns = [c for c in cust_df.columns if c != 'customer_id']

found_conflicts = False
for column_name in other_columns:
    conflicting_data = cust_df.groupBy("customer_id") \
                                .agg(countDistinct(col(column_name)).alias(f"distinct_{column_name}")) \
                                .filter(f"distinct_{column_name} > 1")

    if conflicting_data.count() > 0:
        found_conflicts = True
        print(f"\n⚠️ Conflicts found in column '{column_name}' for customer IDs:")
        conflicting_data.show()
        # Optionally, show full details for these customers:
        # cust_df.join(conflicting_data, "customer_id").orderBy("customer_id").show(truncate=False)

if not found_conflicts:
    print("✅ No conflicting details found for any customer ID across all checked columns.")

To display dupliacte rows in each file

In [0]:
from pyspark.sql.functions import col, count, isnan, when

def run_basic_audit(df_dict):
    for name, df in df_dict.items():
        print(f"\n{'='*20} AUDITING: {name} {'='*20}")

        # ... (Your existing Total Volume and Missing Values checks) ...

        # Check 3: Displaying Duplicate Records
        # Group by all columns and count occurrences of each row
        duplicate_records = df.groupBy(df.columns) \
                              .count() \
                              .filter(col("count") > 1)

        if duplicate_records.count() > 0:
            print(f"⚠️ WARNING: Duplicate records found in {name}:")
            # Show the actual rows that are duplicated
            duplicate_records.show(truncate=False)
        else:
            print(f"✅ No duplicate rows found in {name}.")

run_basic_audit(dfs)

In [0]:
run_basic_audit(dfs)

A. Checking Sales Logic
We want to ensure no one bought "-5" items or got a price of "$0".

keep this for just reference

In [0]:
print("--- Sales Deep Dive ---")
sales_df.select(
    min("quantity").alias("Min_Qty"),
    max("quantity").alias("Max_Qty"),
    min("unit_price").alias("Min_Price"),
    sum(when(col("total_amount") <= 0, 1).otherwise(0)).alias("Invalid_Total_Amounts")
).show()

B. Checking Customer Ages
We want to see if any customer is 0 years old or 150 years old

In [0]:
print("--- Customer Age Check ---")
cust_df.select(
    min("age").alias("Youngest"),
    max("age").alias("Oldest"),
    avg("age").alias("Average_Age")
).show()

C. Checking Product Pricing
Does it ever happen that we sell a product for less than it cost us to buy it? (Cost > Selling Price).

In [0]:
print("--- Product Margin Audit ---")
# Count how many products are sold at a loss (Cost Price > Selling Price)
loss_making_products = prod_df.filter(col("cost_price") > col("selling_price")).count()
print(f"Number of products where Cost Price > Selling Price: {loss_making_products}")

print("\nSample of products where Cost Price > Selling Price:")
prod_df.filter(col("cost_price") > col("selling_price")) \
       .select("product_id", "cost_price", "selling_price").show()

D. Checking Inventory Health
Check for negative stock levels

In [0]:
print("--- Inventory Logic Check ---")
inv_df.filter(col("stock_on_hand") < 0).show()

In [0]:
#here invalid total is 0 but actual case it has be 50 also as price and quantity columns also have 50 such records
#it is solved while cleaning
print("--- Sales Logic Deep Dive ---")
# 1. Check for negative or zero values
sales_df.select(
    count(when(col("quantity") <= 0, 1)).alias("Invalid_Qty"),
    count(when(col("unit_price") <= 0, 1)).alias("Invalid_Price"),
    count(when(col("total_amount") <= 0, 1)).alias("Invalid_Total")
).show()

# this was calculates without considering discount factor

In [0]:
from pyspark.sql.functions import abs, col # Import abs from pyspark.sql.functions

# 2. Out-of-the-box: Check for math inconsistency
# (Allowing for a small rounding difference)
sales_df.withColumn("calculated_amt", col("quantity") * col("unit_price")) \
        .filter(abs(col("calculated_amt") - col("total_amount")) > 1) \
        .select("transaction_id", "total_amount", "calculated_amt") \
        .show(5)

### Corrected Math Inconsistency Check (including Discount)

Let's re-run the consistency check, this time correctly incorporating the `discount` into our calculated total.

here in abs if we put < 1 and check we might get a higher value that depicts values with very small difference that is less than 1 (0.4,0.5....) so we leave it consider for > 1 only


Exactly! In data analysis, it's very common to use a small tolerance (like your > 1 or sometimes > 0.01) when comparing floating-point numbers. This is because computers store and calculate numbers with decimals (like prices and discounts) using floating-point arithmetic, which can introduce tiny, unavoidable rounding errors.

In [0]:
from pyspark.sql.functions import abs, col

# Calculate the number of inconsistent transactions
inconsistent_count = sales_df.withColumn("calculated_total_with_discount",
                                      col("quantity") * col("unit_price") * (1 - col("discount"))) \
                               .filter(abs(col("calculated_total_with_discount") - col("total_amount")) > 2 ) \
                               .count()

print(f"Total number of inconsistent sales transactions: {inconsistent_count}")

while considering the total price I took it along with discount because , 1539 records matched calculated records with total records
but only 358 records are matching with the total_amount without taking the discount

In [0]:
from pyspark.sql.functions import abs, col

# Calculate the number consistent transactions without discount
consistent_count = sales_df.withColumn("calculated_total_without_discount",
                                      col("quantity") * col("unit_price") ) \
                               .filter(abs(col("calculated_total_without_discount") - col("total_amount")) ==0) \
                               .count()

print(f"Total number of consistent sales transactions: {consistent_count}")

///////////////////////////

### Inconsistent Sales Data (where `total_amount` != `quantity * unit_price * (1 - discount)`)

This check identifies transactions where the reported `total_amount` is mathematically inconsistent with the `quantity`, `unit_price`, and `discount` columns, beyond a small rounding tolerance.

this shows all data where our calculated amount not matches the total_amount given


In [0]:
from pyspark.sql.functions import abs, col

inconsistent_sales_data = sales_df.withColumn("calculated_total_with_discount",
                                      col("quantity") * col("unit_price") * (1 - col("discount"))) \
                               .filter(abs(col("calculated_total_with_discount") - col("total_amount")) > 1) \
                               .select("transaction_id", "quantity", "unit_price", "discount", "total_amount", "calculated_total_with_discount")

print(f"Number of inconsistent transactions: {inconsistent_sales_data.count()}")
inconsistent_sales_data.show(20, truncate=False)

This code returns the count of products which was selled at the right amount after applying the discount only 1539 are like that

In [0]:
from pyspark.sql.functions import abs, col

# 2. Out-of-the-box: Check for math inconsistency, now including discount
# (Allowing for a small rounding difference)
exact_match_count = sales_df.withColumn("calculated_total_with_discount",
                    col("quantity") * col("unit_price") * (1 - col("discount"))) \
        .filter(abs(col("calculated_total_with_discount") - col("total_amount")) == 0) \
        .count()

print(f"Number of transactions where calculated total exactly matches actual total: {exact_match_count}")

In [0]:
print("--- Customer Reality Check ---")
# 1. Check for impossible ages
cust_df.filter((col("age") < 18) | (col("age") > 100)).select("customer_id", "age").show()

# 2. Out-of-the-box: Consistency check
# Does a customer appear in Sales but not in Customer Master?
sales_custs = sales_df.select("customer_id").distinct()
missing_custs = sales_custs.join(cust_df, "customer_id", "left_anti").count()
print(f"Transactions from unknown customers: {missing_custs}")

In [0]:
print("--- Margin & Stock Audit ---")
# 1. Negative Margin Check
prod_df.filter(col("selling_price") < col("cost_price")) \
       .select("product_id", "cost_price", "selling_price").show()

# 2. Impossible Stock
inv_df.filter(col("stock_on_hand") < 0).show()

In [0]:
print("--- Platform Standardization Check ---")
click_df.groupBy("platform").count().show()
# Look for: 'Web', 'web', 'WEB' - these should be the same!
#on analyzing it is common only

### Customers with conflicting city details

This code identifies `customer_id`s that have more than one unique city associated with them, indicating potential data inconsistencies where a single customer is recorded with different city details.
so on checking no such fields

In [0]:
from pyspark.sql.functions import countDistinct

conflicting_city_customers = cust_df.groupBy("customer_id") \
                                    .agg(countDistinct("city").alias("distinct_cities")) \
                                    .filter("distinct_cities > 1")

# Show the customer IDs with conflicting city information
print("Customers with conflicting city details (same ID, different cities):")
conflicting_city_customers.show()

# To see the full details of these customers:
print("Full details for customers with conflicting city information:")
cust_df.join(conflicting_city_customers, "customer_id").orderBy("customer_id").show(truncate=False)

A. ID Integrity (The "Ghost" Check)
Check if there are Sales for products or stores that do not exist in your Master files. If a product_id is in Sales but not in Product Master, you won't know the category or cost.

In [0]:
# Check for Sales with missing Product IDs
ghost_products = sales_df.join(prod_df, "product_id", "left_anti").count()
print(f"Sales records with unknown Product IDs: {ghost_products}")

# Check for Sales with missing Store IDs
ghost_stores = sales_df.join(store_df, "store_id", "left_anti").count()
print(f"Sales records with unknown Store IDs: {ghost_stores}")

B. Date Logic (The "Time Traveler" Check)
Ensure that the ingestion_timestamp (when data entered the system) is not before the order_date.

In [0]:
# Count records where ingestion happened before the actual sale
time_travelers = sales_df.filter(col("ingestion_timestamp") < col("order_date")).count()
print(f"Records with impossible date logic: {time_travelers}")

C. Loyalty "None" Check
In your customer_data.csv, the status often shows the word "None". Spark might not see this as a Null because it's a string. Let's see how many "None" strings exist.

In [0]:
cust_df.groupBy("loyalty_status").count().show()

In [0]:
# Store sales_df as a Delta table in capstone_catalog.bronze.sales
sales_df.write.format("delta").mode("overwrite").saveAsTable("capstone_catalog.bronze.sales")

# Store cust_df as a Delta table in capstone_catalog.bronze.customers
cust_df.write.format("delta").mode("overwrite").saveAsTable("capstone_catalog.bronze.customers")

# Store prod_df as a Delta table in capstone_catalog.bronze.products
prod_df.write.format("delta").mode("overwrite").saveAsTable("capstone_catalog.bronze.products")

# Store inv_df as a Delta table in capstone_catalog.bronze.inventory
inv_df.write.format("delta").mode("overwrite").saveAsTable("capstone_catalog.bronze.inventory")

# Store store_df as a Delta table in capstone_catalog.bronze.stores
store_df.write.format("delta").mode("overwrite").saveAsTable("capstone_catalog.bronze.stores")

# Store click_df as a Delta table in capstone_catalog.bronze.clicks
click_df.write.format("delta").mode("overwrite").saveAsTable("capstone_catalog.bronze.clicks")

In [0]:
# Option 1: Reload all DataFrames from Delta tables
# This replaces the CSV-based DataFrames with Delta table-based ones

sales_df = spark.table("capstone_catalog.bronze.sales")
cust_df = spark.table("capstone_catalog.bronze.customers")
prod_df = spark.table("capstone_catalog.bronze.products")
inv_df = spark.table("capstone_catalog.bronze.inventory")
store_df = spark.table("capstone_catalog.bronze.stores")
click_df = spark.table("capstone_catalog.bronze.clicks")

print("✅ All DataFrames now loaded from Delta tables")
print(f"Sales records: {sales_df.count()}")

//////////////////////////////

2. The "Silver Layer" Cleaning Script (The Solution)
This script implements your specific insights. It uses a "Sanity Check" approach to handle the math and the IDs.

In [0]:
# Check what files sales_df is reading from
print("=== Current sales_df source files ===")
csv_sources = sales_df.inputFiles()
print(csv_sources[0] if csv_sources else "No files")

# Check if it's CSV or Delta
if "bronze_volume" in csv_sources[0]:
    print("\n❌ Currently accessing CSV files from volume")
    print("You need to run the 'Load from Delta tables' cell to switch to Delta")
else:
    print("\n✅ Currently accessing Delta tables")

In [0]:
from pyspark.sql import functions as F

# --- 1. CLEANING SALES (Handling your specific findings) ---
# Deduplicate: Only keep unique transactions
cleaned_sales = sales_df.dropDuplicates(["transaction_id"])

# Handling Math Mismatch & Price = 0
# We create a 'final_total' that defaults to 'total_amount' but flags anomalies
#Here I have created column is_anomaly to display true on those records where calculated amount and total amount difference is greater than 2 (because of tax we take >2 only)
cleaned_sales = cleaned_sales.withColumn(
    "is_anomaly",
    F.when(F.abs((F.col("quantity") * F.col("unit_price") * (1 - F.col("discount"))) - F.col("total_amount")) > 2, True).otherwise(False)
)

# Handling Quantity = -1 (The Return Logic)
#here I have created a new column transaction_type to give value as return on place where quantity is negative
cleaned_sales = cleaned_sales.withColumn(
    "transaction_type",
    F.when(F.col("quantity") < 0, "RETURN").otherwise("SALE")
)

# --- 2. CLEANING CUSTOMERS (Handling Null Ages and "None" Loyalty Status) ---
# Calculate median age for filling nulls
median_age = cust_df.approxQuantile("age", [0.5], 0.01)[0]

# First, fill null/NaN ages and any actual null loyalty_status with 'Standard'
temp_cleaned_customers = cust_df.na.fill({"age": median_age, "loyalty_status": "Standard"})

# Then, standardize the 'None' string in loyalty_status
cleaned_customers = temp_cleaned_customers.withColumn(
    "loyalty_status",
    F.when(F.col("loyalty_status") == "None", "Standard")
     .otherwise(F.col("loyalty_status"))
)

# --- 3. CLEANING INVENTORY (Handling Negative Stock) ---
# here I have made stock_on_hand 0 and also created a flagging to show those records which had a negative value

cleaned_inventory = inv_df.withColumn(
    "inventory_sync_error",
    F.when(F.col("stock_on_hand") < 0, True).otherwise(False)
).withColumn(
    "stock_on_hand",
    F.when(F.col("stock_on_hand") < 0, 0).otherwise(F.col("stock_on_hand"))
)
# --- 4. SANITY CHECK: The "Orphan" Filter ---
# Ensure every product sold actually exists in our Product Master
#here It takes all records from cleaned_sales and tries to find a matching record in prod_df based on the common column product_id.
#Only the rows where a product_id exists in both cleaned_sales and prod_df will be included in the resulting valid_sales DataFrame.
#on checking it is already in the right state only that all records sold are in product master , no issues and no cleaning is done
valid_sales = cleaned_sales.join(prod_df, "product_id", "inner")

print("Cleaning complete. Data is now 'Silver' standard.")

In [0]:
#here I have checked if all products sold are there in product master , yes it is so no issue
#this code doesnt affect our data
print("--- Checking for dropped sales records due to 'Orphan' Products ---")

initial_sales_count = cleaned_sales.count()
valid_sales_count = valid_sales.count()

if initial_sales_count > valid_sales_count:
    dropped_records = initial_sales_count - valid_sales_count
    print(f"⚠️ WARNING: {dropped_records} sales records were dropped because their product_id was not found in the Product Master.")
    print(f"Initial cleaned_sales count: {initial_sales_count}")
    print(f"Valid sales count (after join with products): {valid_sales_count}")

    # Optionally, to show some of the dropped records:
    # dropped_sales_sample = cleaned_sales.join(prod_df, "product_id", "left_anti")
    # if dropped_sales_sample.count() > 0:
    #     print("\nSample of dropped sales records:")
    #     dropped_sales_sample.show(5, truncate=False)
else:
    print(f"✅ No sales records were dropped. All {initial_sales_count} records have a matching product in the Product Master.")

In [0]:
#print(f"The median age calculated and used to fill nulls was: {median_age}")

# Find customers with null ages in the original DataFrame
# Note: From the initial audit, the '200' missing values for age were actually `null` or `NaN` values that Spark recognized as missing.
customers_with_original_null_age = cust_df.filter(col("age").isNull() | isnan(col("age"))).select("customer_id").collect()

if customers_with_original_null_age:
    print("\n--- Original Customer Data (with null/NaN ages) ---")
    customer_ids_to_show = [row.customer_id for row in customers_with_original_null_age]
    cust_df.filter(col("customer_id").isin(customer_ids_to_show)).show()

    print("\n--- Cleaned Customer Data (with ages filled by median) ---")
    cleaned_customers.filter(col("customer_id").isin(customer_ids_to_show)).show()
else:
    print("\nNo customers with original null/NaN ages were found to display (which means our initial audit might have caught non-standard nulls or it's already clean here).")


In [0]:
#here I have now how many sales records are there after cleaning
from pyspark.sql.functions import countDistinct, col

print("--- Checking for duplicate transaction_id in cleaned_sales ---")

total_cleaned_sales_rows = cleaned_sales.count()
distinct_transaction_ids = cleaned_sales.select(countDistinct("transaction_id")).collect()[0][0]

if total_cleaned_sales_rows == distinct_transaction_ids:
    print(f"✅ No duplicate transaction_ids found in cleaned_sales. All {total_cleaned_sales_rows} rows have unique transaction IDs.")
else:
    print(f"⚠️ WARNING: Duplicate transaction_ids still exist in cleaned_sales. Found {distinct_transaction_ids} distinct IDs out of {total_cleaned_sales_rows} total rows.")

Note now out 10000500 records now only 1000000 are there

In [0]:
print("--- Counting 'None' loyalty statuses in cleaned_customers ---")
none_loyalty_count = cleaned_customers.filter(col("loyalty_status") == "None").count()
print(f"Number of customers with 'None' loyalty status: {none_loyalty_count}")

The Decision: We treat "None" as a "Standard" customer. We assume that if they aren't marked as Gold, Silver, or Bronze, they are a regular walk-in customer with no special privileges.

In [0]:
# This cell's functionality for cleaning loyalty_status has been moved into wMeej4f4684B for consistency.
# from pyspark.sql import functions as F

# # Fix the Customer Master first
# cleaned_customers = cust_df.withColumn(
#     "loyalty_status",
#     F.when(F.col("loyalty_status").isNull() | (F.col("loyalty_status") == "None"), "Standard")
#      .otherwise(F.col("loyalty_status"))
# )

In [0]:
print("--- Counting 'None' loyalty statuses in cleaned_customers ---")
none_loyalty_count = cleaned_customers.filter(col("loyalty_status") == "None").count()
print(f"Number of customers with 'None' loyalty status: {none_loyalty_count}")

Now I have tried to replace all unit price 0 fields with I check with the original price from another file or I fill with the averge
details given below

In [0]:
from pyspark.sql import functions as F

# Calculate average selling price per category for fallback
avg_selling_price_per_category = prod_df.groupBy("category") \
                                        .agg(F.avg("selling_price").alias("avg_category_selling_price"))

# Step 1: Join Sales with Product Master to get "Official" price and "category"
sales_with_prices = cleaned_sales.join(
    prod_df.select("product_id", "selling_price", "category"), # Include 'category' here
    on="product_id",
    how="left"
).withColumnRenamed("selling_price", "master_selling_price")

# Join with cleaned_customers to bring in loyalty_status
sales_with_customer_info = sales_with_prices.join(
    cleaned_customers.select("customer_id", "loyalty_status"),
    on="customer_id",
    how="left"
)

# Join with average selling price per category
sales_with_customer_info = sales_with_customer_info.join(
    avg_selling_price_per_category,
    on="category",
    how="left"
)

# Step 2: Apply the Recovery Logic with revised priority
cleaned_sales_final = sales_with_customer_info.withColumn(
    "final_unit_price",
    F.when(F.col("unit_price") > 0, F.col("unit_price")) # 1. Keep price if original is valid (>0)
     .when((F.col("unit_price") <= 0) & (F.col("master_selling_price").isNotNull()) & (F.col("master_selling_price") > 0), F.col("master_selling_price")) # 2. If original was problematic (<=0), use master_selling_price if valid
     .when((F.col("unit_price") <= 0) & (F.col("loyalty_status") == "Gold"), 0.0) # 3. If original was problematic and no valid master price, check if Gold customer (promotion)
     .otherwise(F.col("avg_category_selling_price")) # 4. Fallback to average category selling price
)

# Step 3: Recalculate Total Amount based on fixed price
cleaned_sales_final = cleaned_sales_final.withColumn(
    "total_amount",
    F.col("quantity") * F.col("final_unit_price") * (1 - F.col("discount"))
)

In [0]:
print("--- Sample of Cleaned Sales Final Data ---")
cleaned_sales_final.select(
    "transaction_id",
    "customer_id",
    "loyalty_status",
    "product_id",
    "quantity",
    "unit_price", # Original unit price
    "master_selling_price", # Selling price from product master
    "final_unit_price", # The unit price after applying recovery logic
    "discount",
    "total_amount" # The recalculated total amount
).show(20, truncate=False)

1. The Strategy: "The 3-Step Price Recovery"
Instead of just calling it an error, we follow this hierarchy to fix the $0.0 values:

Lookup: Check the Product_Master table for the correct selling_price for that product_id.



Default: If neither of the above applies, use the Average Selling Price for that category so the revenue isn't under-reported.

this was done to clean data with price 0



Now lets display those fields

In [0]:
print("--- Records with Original unit_price = 0 ---")

original_zero_price_transactions = sales_df.filter(col("unit_price") == 0)
original_count = original_zero_price_transactions.count()

print(f"Number of transactions with original unit_price = 0: {original_count}")

if original_count > 0:
    print("\n--- Sample of Original Records with unit_price = 0 ---")
    original_zero_price_transactions.show(20, truncate=False)

    # Now, let's see how these specific transactions appear in cleaned_sales_final
    print("\n--- Transformation of these Records in Cleaned Sales Final ---")
    transformed_zero_price_transactions = cleaned_sales_final.join(
        original_zero_price_transactions.select("transaction_id"),
        "transaction_id",
        "inner"
    ).select(
        "transaction_id",
        "customer_id",
        "loyalty_status", # To see why final_unit_price might be 0
        "product_id",
        "quantity",
        "unit_price", # Original unit price
        "master_selling_price", # Selling price from product master
        "final_unit_price", # The unit price after applying recovery logic
        "discount",
        "total_amount" # The recalculated total amount
    )
    transformed_zero_price_transactions.show(20, truncate=False)
else:
    print("No transactions found with an original unit_price of 0.")

In [0]:
#I tried to display all cleaning done to our sales (cleaned_sales_final)
print("--- All Records of cleaned_sales_final ---")
cleaned_sales_final.show(truncate=False)

In [0]:
#note that under unit price , we created final_unit_price which has updated values
print("--- Rows in cleaned_sales_final with unit_price = 0 ---")
cleaned_sales_final.filter(col("unit_price") == 0).show(truncate=False)

In [0]:
from pyspark.sql.functions import col, abs, when

print("--- Adding 'calculated_total_amount' and 'is_anomaly_recalculated' to cleaned_sales_final ---")

# 1. Add a column 'calculated_total_amount' using quantity, original unit_price, and discount
cleaned_sales_final = cleaned_sales_final.withColumn(
    "calculated_total_amount",
    col("quantity") * col("unit_price") * (1 - col("discount"))
)

# 2. Compare this new calculated_total_amount with the existing total_amount
#    and create a new 'is_anomaly_recalculated' column.
#    We'll use the same anomaly threshold (> 2) as previously defined.
cleaned_sales_final = cleaned_sales_final.withColumn(
    "is_anomaly_recalculated",
    when(abs(col("calculated_total_amount") - col("total_amount")) > 2, True).otherwise(False)
)

# Display relevant columns to show the new calculated values and anomaly flag
print("Sample of cleaned_sales_final with new calculated_total_amount and is_anomaly_recalculated:")
cleaned_sales_final.select(
    "transaction_id",
    "quantity",
    "unit_price", # Original unit price
    "final_unit_price", # Cleaned/fixed unit price
    "discount",
    "calculated_total_amount", # Calculated using original unit_price
    "total_amount", # Recalculated total amount using final_unit_price
    "is_anomaly", # Original anomaly flag (based on original unit_price and original total_amount)
    "is_anomaly_recalculated" # New anomaly flag (based on calculated_total_amount and current total_amount)
).show(20, truncate=False)

print(f"Total records with is_anomaly (original) = True: {cleaned_sales_final.filter(col('is_anomaly') == True).count()}")
print(f"Total records with is_anomaly_recalculated = True: {cleaned_sales_final.filter(col('is_anomaly_recalculated') == True).count()}")

In [0]:
print("--- Checking is_anomaly_recalculated records ---")
inconsistent_recalculated = cleaned_sales_final.filter(col("is_anomaly_recalculated") == True)

print(f"Number of records with is_anomaly_recalculated = True: {inconsistent_recalculated.count()}")

if inconsistent_recalculated.count() > 0:
    print("\nSample of records with is_anomaly_recalculated = True:")
    inconsistent_recalculated.select(
        "transaction_id",
        "quantity",
        "unit_price", # Original unit price
        "final_unit_price", # Cleaned/fixed unit price
        "discount",
        "calculated_total_amount", # Calculated using original unit_price
        "total_amount", # Recalculated total amount using final_unit_price
        "is_anomaly_recalculated"
    ).show(20, truncate=False)
else:
    print("No records found with is_anomaly_recalculated = True, indicating consistency after final cleaning.")

In [0]:
#out original unit price is same only
#changes have been done to final_unit_price
print("--- Rows in cleaned_sales_final with unit_price = 0 ---")
cleaned_sales_final.filter(col("unit_price") == 0).count()

Code to check total_amount negative

In [0]:
print("--- Checking for negative total_amount in original sales_df ---")
negative_total_original = sales_df.filter(col("total_amount") < 0).count()
print(f"Number of records with negative total_amount in original sales_df: {negative_total_original}")
if negative_total_original > 0:
    sales_df.filter(col("total_amount") < 0).show(truncate=False)

print("\n--- Checking for negative total_amount in cleaned_sales_final ---")
negative_total_cleaned = cleaned_sales_final.filter(col("total_amount") < 0).count()
print(f"Number of records with negative total_amount in cleaned_sales_final: {negative_total_cleaned}")
if negative_total_cleaned > 0:
    cleaned_sales_final.filter(col("total_amount") < 0).show(truncate=False)

command to shwo negative quantity

In [0]:
print("--- Counting records with negative quantity in cleaned_sales_final ---")
negative_quantity_count = cleaned_sales_final.filter(col("quantity") < 0).count()
print(f"Number of records with negative quantity: {negative_quantity_count}")

if negative_quantity_count > 0:
    print("\nSample of records with negative quantity:")
    cleaned_sales_final.filter(col("quantity") < 0).show(truncate=False)

Now lets try to get the output of same code to check for invalid records, the output will be same but actual validation and cleaning makes it remain same for further analysis

In [0]:
print("--- Sales Logic Deep Dive ---")
# 1. Check for negative or zero values
cleaned_sales_final.select(
    count(when(col("quantity") <= 0, 1)).alias("Invalid_Qty"),
    count(when(col("unit_price") <= 0, 1)).alias("Invalid_Price"),
    count(when(col("total_amount") <= 0, 1)).alias("Invalid_Total")
).show()

### 1. Validated 'RETURN' Transactions (originally negative quantity)

In [0]:
print("--- Sample of validated 'RETURN' transactions ---")
display(
    cleaned_sales_final.filter(col("transaction_type") == "RETURN")
        .select("transaction_id", "customer_id", "quantity", "unit_price", "discount", "total_amount", "transaction_type")
)

print(f"Total validated 'RETURN' transactions: {cleaned_sales_final.filter(col('transaction_type') == 'RETURN').count()}")

### 2. Validated 'Gold Customer' Promotions (originally zero unit price)

In [0]:
print("--- Sample of validated 'Gold Customer' zero-price promotions ---")
display(
    cleaned_sales_final.filter((col("loyalty_status") == "Gold") & (col("final_unit_price") == 0))
        .select("transaction_id", "customer_id", "loyalty_status", "quantity", "unit_price", "final_unit_price", "discount", "total_amount")
)

print(f"Total validated 'Gold Customer' zero-price promotions: {cleaned_sales_final.filter((col('loyalty_status') == 'Gold') & (col('final_unit_price') == 0)).count()}")

Display all the records with total price <=0


In [0]:
print("--- Records in cleaned_sales_final with total_amount <= 0 ---")
cleaned_sales_final.filter(col("total_amount") <= 0).show(truncate=False)

In [0]:
print("--- Records in cleaned_sales_final with total_amount <= 0 ---")
cleaned_sales_final.filter(col("total_amount") <= 0).count()

In [0]:
print("--- Records in cleaned_sales_final with final_unit_price = 0 ---")
cleaned_sales_final.filter(col("final_unit_price") == 0).show(truncate=False)

LETS ANALYSE PRODUCTS WITH SP>CP



In [0]:
print("--- Product Margin Audit ---")
# Count how many products are sold at a loss (Cost Price > Selling Price)
loss_making_products = prod_df.filter(col("cost_price") > col("selling_price")).count()
print(f"Number of products where Cost Price > Selling Price: {loss_making_products}")

print("\nSample of products where Cost Price > Selling Price:")
prod_df.filter(col("cost_price") > col("selling_price")) \
       .select("product_id", "cost_price", "selling_price").show()

In [0]:
print("--- Products where Selling Price > Cost Price ---")
profitable_products_count = prod_df.filter(col("selling_price") > col("cost_price")).count()
print(f"Number of products where Selling Price > Cost Price: {profitable_products_count}")

print("\nSample of products where Selling Price > Cost Price:")
prod_df.filter(col("selling_price") > col("cost_price")) \
       .select("product_id", "cost_price", "selling_price").show(20, truncate=False)

In [0]:
print("--- Products where Selling Price == Cost Price ---")
equal_price_products_count = prod_df.filter(col("selling_price") == col("cost_price")).count()
print(f"Number of products where Selling Price == Cost Price: {equal_price_products_count}")

print("\nSample of products where Selling Price == Cost Price:")
prod_df.filter(col("selling_price") == col("cost_price")) \
       .select("product_id", "cost_price", "selling_price").show(20, truncate=False)

# Task
Create a new boolean column `is_loss_making` in the `prod_df` DataFrame, setting it to `True` where `cost_price` is greater than `selling_price`, and `False` otherwise, to flag and track loss-making products.

## Identify Loss-Making Products

### Subtask:
Reconfirm and identify the products from `prod_df` where `cost_price` is greater than `selling_price`. This step is to ensure we are targeting the correct records.


**Reasoning**:
The subtask requires reconfirming and identifying loss-making products. I will filter `prod_df` where `cost_price` is greater than `selling_price`, count them, and display relevant columns, as requested in the instructions.



In [0]:
print("--- Reconfirming Loss-Making Products ---")

# 1. Filter the prod_df DataFrame to include only rows where cost_price > selling_price
loss_making_products_df = prod_df.filter(col("cost_price") > col("selling_price"))

# 2. Count the number of such products and print it
loss_making_count = loss_making_products_df.count()
print(f"Number of products where Cost Price > Selling Price (reconfirmed): {loss_making_count}")

# 3. Display the product_id, cost_price, and selling_price for these loss-making products
print("\nProduct details where Cost Price > Selling Price (reconfirmed):")
loss_making_products_df.select("product_id", "cost_price", "selling_price").show(truncate=False)

# Task
Add a new boolean column named `is_loss_making` to the `prod_df` DataFrame. This column should be `True` if `cost_price` is greater than `selling_price`, and `False` otherwise. Afterward, display a sample of the updated `prod_df` to verify the addition of the new column.

## Add is_loss_making Column

### Subtask:
Create a new boolean column named `is_loss_making` in the `prod_df` DataFrame. This column will be set to `True` where `cost_price` is greater than `selling_price`, and `False` otherwise.


**Reasoning**:
The subtask requires adding a new boolean column `is_loss_making` to `prod_df` based on the comparison of `cost_price` and `selling_price`. I will use `withColumn` and `when` functions to create this column.



In [0]:
from pyspark.sql import functions as F

# Add the 'is_loss_making' column to prod_df
prod_df = prod_df.withColumn(
    "is_loss_making",
    F.when(F.col("cost_price") > F.col("selling_price"), True).otherwise(False)
)

print("prod_df with 'is_loss_making' column:")
prod_df.select("product_id", "cost_price", "selling_price", "is_loss_making").show(15)

## Final Task

### Subtask:
Summarize the action taken and confirm the successful addition of the `is_loss_making` column.


## Summary:

### Q&A
The `is_loss_making` column was successfully added to the `prod_df` DataFrame, and its values were correctly determined based on the comparison between `cost_price` and `selling_price`.

### Data Analysis Key Findings
*   A new boolean column named `is_loss_making` was successfully added to the `prod_df` DataFrame.
*   The `is_loss_making` column correctly evaluates to `True` when `cost_price` is greater than `selling_price`, and `False` otherwise.
*   The sample output displayed, where `is_loss_making` was consistently `false` for the shown rows, confirmed the correct application of the defined logic.

### Insights or Next Steps
*   The `prod_df` DataFrame is now enriched with a critical indicator for profitability analysis.
*   The next step could involve analyzing the distribution of `is_loss_making` products to identify the proportion of products that are currently loss-making.


done

In [0]:
print("--- Checking is_anomaly_recalculated records ---")
inconsistent_recalculated = cleaned_sales_final.filter(col("is_anomaly_recalculated") == True)

print(f"Number of records with is_anomaly_recalculated = True: {inconsistent_recalculated.count()}")

if inconsistent_recalculated.count() > 0:
    print("\nSample of records with is_anomaly_recalculated = True:")
    inconsistent_recalculated.select(
        "transaction_id",
        "quantity",
        "unit_price", # Original unit price
        "final_unit_price", # Cleaned/fixed unit price
        "discount",
        "calculated_total_amount", # Calculated using original unit_price
        "total_amount", # Recalculated total amount using final_unit_price
        "is_anomaly_recalculated"
    ).show(20, truncate=False)
else:
    print("No records found with is_anomaly_recalculated = True, indicating consistency after final cleaning.")

In [0]:
from pyspark.sql.functions import col, abs, when

print("--- Final Consistency Check (using final_unit_price) ---")

# 1. Calculate the expected total amount using the final_unit_price
cleaned_sales_final_check = cleaned_sales_final.withColumn(
    "final_calculated_total_amount",
    col("quantity") * col("final_unit_price") * (1 - col("discount"))
)

# 2. Create a new anomaly flag by comparing this calculated amount with the existing total_amount
#    Using a small tolerance for floating-point comparisons
cleaned_sales_final_check = cleaned_sales_final_check.withColumn(
    "is_anomaly_final_consistency",
    when(abs(col("final_calculated_total_amount") - col("total_amount")) > 0.01, True).otherwise(False)
)

# 3. Count records where this new anomaly flag is True
inconsistent_final_count = cleaned_sales_final_check.filter(col("is_anomaly_final_consistency") == True).count()

print(f"Number of records with is_anomaly_final_consistency = True: {inconsistent_final_count}")

if inconsistent_final_count > 0:
    print("\nSample of records with is_anomaly_final_consistency = True:")
    cleaned_sales_final_check.filter(col("is_anomaly_final_consistency") == True).select(
        "transaction_id",
        "quantity",
        "unit_price", # Original unit price
        "final_unit_price", # Cleaned/fixed unit price
        "discount",
        "total_amount", # Recalculated total amount using final_unit_price
        "final_calculated_total_amount",
        "is_anomaly_final_consistency"
    ).show(20, truncate=False)
else:
    print("✅ All records are consistent with final_unit_price after cleaning (within a tolerance of 0.01).")

In [0]:
print(f"Number of records with is_anomaly_final_consistency = True: {cleaned_sales_final_check.filter(col('is_anomaly_final_consistency') == True).count()}")

In [0]:
cleaned_sales_final_check.show()


CLEANING DONE


RECHECKING CLEANED DATA

In [0]:
from pyspark.sql.types import DoubleType, FloatType
from pyspark.sql.functions import col, count, when, isnan

def run_basic_audit(df_dict):
    for name, df in df_dict.items():
        print(f"\n{'='*20} AUDITING: {name} {'='*20}")

        # Check 1: Total Volume
        total_rows = df.count()
        print(f"Total Records: {total_rows}")

        # Check 2: Missing Values (Nulls)
        # This scans every column and counts how many empty spots exist
        print("Missing Values per Column:")
        missing_value_expressions = []
        for c in df.columns:
            # Check if the column is of a numeric type (DoubleType or FloatType)
            if isinstance(df.schema[c].dataType, (DoubleType, FloatType)):
                missing_value_expressions.append(count(when(isnan(col(c)) | col(c).isNull(), c)).alias(c))
            else:
                missing_value_expressions.append(count(when(col(c).isNull(), c)).alias(c))
        df.select(missing_value_expressions).show()

        # Check 3: Duplicates
        unique_rows = df.dropDuplicates().count()
        if total_rows > unique_rows:
            print(f"⚠️ WARNING: Found {total_rows - unique_rows} duplicate rows!")
        else:
            print("✅ No duplicate rows found.")

# Create a new dictionary with the latest variable names
latest_dataframes = {
    "Sales": cleaned_sales_final_check,
    "Products": prod_df,
    "Stores": store_df,
    "Customers": cleaned_customers,
    "Inventory": cleaned_inventory,
    "Clickstream": click_df
}

run_basic_audit(latest_dataframes)

In [0]:
from pyspark.sql.types import IntegerType, DoubleType, FloatType, LongType, ShortType, ByteType
from pyspark.sql.functions import col, count, when, isnan # Added for completeness as 'col' is used

print("--- Checking for Negative Values in Numeric Columns (on Cleaned Data) ---")

# Re-defining latest_dataframes for standalone execution, referencing cleaned DataFrames
latest_dataframes = {
    "Sales": cleaned_sales_final_check,
    "Products": prod_df,
    "Stores": store_df,
    "Customers": cleaned_customers,
    "Inventory": cleaned_inventory,
    "Clickstream": click_df
}

for name, df in latest_dataframes.items():
    print(f"\n{'='*20} Checking DataFrame: {name} {'='*20}")
    found_negative = False
    for col_name in df.columns:
        # Check if the column is a numeric type
        if isinstance(df.schema[col_name].dataType, (IntegerType, DoubleType, FloatType, LongType, ShortType, ByteType)):
            negative_values_df = df.filter(col(col_name) < 0)
            if negative_values_df.count() > 0:
                found_negative = True
                print(f"\n⚠️ Negative values found in column '{col_name}':")
                negative_values_df.show(truncate=False)

    if not found_negative:
        print(f"✅ No negative values found in any numeric column of {name}.")

In [0]:
from pyspark.sql.types import DoubleType, FloatType, IntegerType
from pyspark.sql.functions import col, count, when, isnan, min, max

def run_basic_audit(df_dict):
    for name, df in df_dict.items():
        print(f"\n{'='*20} AUDITING: {name} {'='*20}")

        # Check 1: Total Volume
        total_rows = df.count()
        print(f"Total Records: {total_rows}")

        # Check 2: Missing Values (Nulls)
        print("Missing Values per Column:")
        missing_value_expressions = []
        for c in df.columns:
            if isinstance(df.schema[c].dataType, (DoubleType, FloatType)):
                missing_value_expressions.append(count(when(isnan(col(c)) | col(c).isNull(), c)).alias(c))
            else:
                missing_value_expressions.append(count(when(col(c).isNull(), c)).alias(c))
        df.select(missing_value_expressions).show()

        # Check 3: Duplicates
        unique_rows = df.dropDuplicates().count()
        if total_rows > unique_rows:
            print(f"⚠️ WARNING: Found {total_rows - unique_rows} duplicate rows!")
        else:
            print("✅ No duplicate rows found.")

        # --- Extended Checks for Cleaned Data --- (reflecting 'latest variables')
        if name == "Sales": # This refers to cleaned_sales_final_check
            # Verify final consistency check results
            if "is_anomaly_final_consistency" in df.columns:
                final_inconsistent_count = df.filter(col("is_anomaly_final_consistency") == True).count()
                if final_inconsistent_count == 0:
                    print("✅ Sales: All records are consistent after final cleaning (is_anomaly_final_consistency = False).")
                else:
                    print(f"⚠️ Sales: Found {final_inconsistent_count} records with final consistency anomalies.")

            # Confirm expected negative quantities for returns
            negative_qty_count = df.filter(col("quantity") < 0).count()
            if negative_qty_count > 0:
                print(f"ℹ️ Sales: {negative_qty_count} records have negative quantity (expected for RETURN transactions).")

            # Confirm expected negative total_amount for returns
            negative_total_count = df.filter(col("total_amount") < 0).count()
            if negative_total_count > 0:
                print(f"ℹ️ Sales: {negative_total_count} records have negative total_amount (expected for RETURN transactions).")

        elif name == "Customers": # This refers to cleaned_customers
            # Verify null ages were filled
            null_age_count = df.filter(col("age").isNull() | isnan(col("age"))).count()
            if null_age_count == 0:
                print("✅ Customers: No null/NaN ages found (values were filled).")
            else:
                print(f"⚠️ Customers: Found {null_age_count} null/NaN ages.")

            # Verify 'None' loyalty status was handled
            none_loyalty_count = df.filter(col("loyalty_status") == "None").count()
            if none_loyalty_count == 0:
                print("✅ Customers: No 'None' loyalty statuses found (values were standardized).")
            else:
                print(f"⚠️ Customers: Found {none_loyalty_count} 'None' loyalty statuses.")

        elif name == "Inventory": # This refers to cleaned_inventory
            # Verify negative stock was handled
            negative_stock_count = df.filter(col("stock_on_hand") < 0).count()
            if negative_stock_count == 0:
                print("✅ Inventory: No negative stock_on_hand values found (values were set to 0).")
            else:
                print(f"⚠️ Inventory: Found {negative_stock_count} negative stock_on_hand values.")

        elif name == "Products": # This refers to prod_df
            # Verify is_loss_making column exists and its counts
            if "is_loss_making" in df.columns:
                loss_making_count = df.filter(col("is_loss_making") == True).count()
                print(f"ℹ️ Products: Found {loss_making_count} products flagged as 'is_loss_making'.")
            else:
                print("⚠️ Products: 'is_loss_making' column not found.")

In [0]:
# Create a new dictionary with the latest variable names for the audit
latest_dataframes = {
    "Sales": cleaned_sales_final_check,
    "Products": prod_df,
    "Stores": store_df,
    "Customers": cleaned_customers,
    "Inventory": cleaned_inventory,
    "Clickstream": click_df
}

# Run the basic audit on the latest cleaned dataframes
run_basic_audit(latest_dataframes)

In [0]:
print("--- Diagnostic Check: Null/NaN Ages in cleaned_customers ---")
from pyspark.sql.functions import col, isnan

null_ages_in_cleaned_customers = cleaned_customers.filter(col("age").isNull() | isnan(col("age")))
null_ages_count = null_ages_in_cleaned_customers.count()

print(f"Number of null/NaN ages found in cleaned_customers: {null_ages_count}")
if null_ages_count > 0:
    print("Sample of cleaned_customers with null/NaN ages:")
    null_ages_in_cleaned_customers.show(5, truncate=False)
else:
    print("✅ No null/NaN ages found in cleaned_customers after cleaning.")

In [0]:
print("--- Customer Age Check (on Cleaned Data) ---")
cleaned_customers.select(
    min("age").alias("Youngest"),
    max("age").alias("Oldest"),
    avg("age").alias("Average_Age")
).show()

In [0]:
print("--- Inventory Logic Check (on Cleaned Data) ---")
cleaned_inventory.filter(col("stock_on_hand") < 0).show()

In [0]:
print(f"Number of records with is_anomaly_final_consistency = True: {cleaned_sales_final_check.filter(col('is_anomaly_final_consistency') == True).count()}")

In [0]:
cleaned_customers.groupBy("loyalty_status").count().show()

It converts the event_timestamp column into a proper timestamp data type. This is crucial for accurate time-based analysis, filtering, and sorting, as it ensures Spark recognizes the column's values as actual dates and times rather than just strings

if platforms were recorded as 'web', 'Web', or 'WEB', they will all become 'WEB'. This ensures consistency and makes it easier to group and analyze data by platform.

In [0]:
# Silver Layer: Inventory Alert System
cleaned_inventory = cleaned_inventory.withColumn(
    "reorder_required",
    F.when(F.col("stock_on_hand") < F.col("reorder_level"), True).otherwise(False)
)

In [0]:
# Silver Layer: Clickstream Standardization
cleaned_clickstream = click_df.withColumn(
    "platform", F.upper(F.col("platform"))
).withColumn(
    "event_timestamp", F.to_timestamp(F.col("event_timestamp"))
)

In [0]:
import pyspark.sql.functions as F

anomaly_count = cleaned_sales_final_check.filter(F.col('is_anomaly_final_consistency') == True).count()

print(f"Raw Sales: {sales_df.count()} | Cleaned Sales: {cleaned_sales_final.count()}")
print(f"Anomalies Found & Fixed: {anomaly_count}")
print(f"Returns Identified: {cleaned_sales_final.filter(F.col('transaction_type')=='RETURN').count()}")

### Products Requiring Reorder

This list identifies `product_id` and `store_id` combinations where `stock_on_hand` is below `reorder_level`, indicating that an order needs to be placed for these items to replenish stock.

In [0]:
print("--- List of Products and Stores Requiring Reorder ---")
reorder_list_df = cleaned_inventory.filter(col("reorder_required") == True) \
                                  .select("product_id", "store_id", "stock_on_hand", "reorder_level")

reorder_list_df.show(truncate=False)

print(f"Total unique products requiring reorder: {reorder_list_df.select('product_id').distinct().count()}")
print(f"Total instances (product-store combinations) requiring reorder: {reorder_list_df.count()}")

In [0]:
print("--- Records where 'reorder_required' is True ---")
cleaned_inventory.filter(col("reorder_required") == True).show(truncate=False)

In [0]:
from pyspark.sql.functions import col, isnan

# Cast the 'age' column to IntegerType
cleaned_customers = cleaned_customers.withColumn("age", col("age").cast("integer"))

# Count non-null/NaN ages
non_null_age_count = cleaned_customers.filter(col("age").isNotNull() & ~isnan(col("age"))).count()

print(f"The 'age' column has been converted to integer type.")
print(f"Total count of non-null/NaN ages: {non_null_age_count}")

print("\nSample of cleaned_customers with updated 'age' column:")
cleaned_customers.select("customer_id", "age", "gender").show(5)

In [0]:
cleaned_sales = cleaned_sales_final_check.select(
    "transaction_id",
    "order_date",
    "channel",
    "store_id",
    "product_id",
    "customer_id",
    "quantity",
    "final_unit_price", # The cleaned and corrected unit price
    "discount",
    "total_amount",     # The final, corrected total amount
    "payment_type",
    "ingestion_timestamp",
    "category",
    "transaction_type",
    "loyalty_status"
)

print("--- Gold Layer Sales Data Sample (cleaned_sales) ---")
cleaned_sales.show(5, truncate=False)
print(f"\nGold Sales DataFrame schema after selection: ")
cleaned_sales.printSchema()

### Checking for `total_amount` Inconsistencies and Invalid `final_unit_price` in `cleaned_sales`

In [0]:
from pyspark.sql.functions import col, abs, when

print("--- 1. Checking `total_amount` Consistency in cleaned_sales ---")

# Calculate the expected total amount using the final_unit_price from cleaned_sales
cleaned_sales_check = cleaned_sales.withColumn(
    "calculated_total_amount_check",
    col("quantity") * col("final_unit_price") * (1 - col("discount"))
)

# Create an anomaly flag by comparing this calculated amount with the existing total_amount
# Using a small tolerance for floating-point comparisons (e.g., 0.01)
inconsistent_total_amount = cleaned_sales_check.filter(
    abs(col("calculated_total_amount_check") - col("total_amount")) > 0.01
)

inconsistent_total_count = inconsistent_total_amount.count()
print(f"Number of records with `total_amount` inconsistency: {inconsistent_total_count}")

if inconsistent_total_count > 0:
    print("\nSample of records with `total_amount` inconsistency:")
    inconsistent_total_amount.select(
        "transaction_id", "quantity", "final_unit_price", "discount",
        "total_amount", "calculated_total_amount_check"
    ).show(20, truncate=False)
else:
    print("✅ No `total_amount` inconsistencies found (within 0.01 tolerance).")

print("\n--- 2. Checking for Zero or Negative `final_unit_price` ---")

invalid_final_unit_price = cleaned_sales.filter(col("final_unit_price") <= 0)
invalid_final_unit_price_count = invalid_final_unit_price.count()

print(f"Number of records with zero or negative `final_unit_price`: {invalid_final_unit_price_count}")

if invalid_final_unit_price_count > 0:
    print("\nSample of records with zero or negative `final_unit_price`:")
    invalid_final_unit_price.select(
        "transaction_id", "product_id", "quantity", "final_unit_price", "total_amount"
    ).show(20, truncate=False)
else:
    print("✅ No zero or negative `final_unit_price` values found.")

### Latest `cleaned_sales` DataFrame (Gold Layer)

In [0]:
print("--- Sample of cleaned_sales (Gold Layer Sales Data) ---")
cleaned_sales.show(5, truncate=False)

### Latest `prod_df` DataFrame (with `is_loss_making`)

In [0]:
print("--- Sample of prod_df (Product Data with is_loss_making) ---")
prod_df.show(5, truncate=False)

### Latest `store_df` DataFrame

In [0]:
print("--- Sample of store_df (Store Data) ---")
store_df.show(5, truncate=False)

### Latest `cleaned_customers` DataFrame

In [0]:
print("--- Sample of cleaned_customers (Customer Data) ---")
cleaned_customers.show(5, truncate=False)

### Latest `cleaned_inventory` DataFrame (with `reorder_required`)

In [0]:
print("--- Sample of cleaned_inventory (Inventory Data) ---")
cleaned_inventory.show(5, truncate=False)

In [0]:
cleaned_inventory.filter(col("inventory_sync_error")==True).count()

### Latest `cleaned_clickstream` DataFrame

In [0]:
print("--- Sample of cleaned_clickstream (Clickstream Data) ---")
cleaned_clickstream.show(5, truncate=False)

In [0]:
# Assign existing DataFrames to new variables with 'cleaned_' prefix
# Note: cleaned_sales, cleaned_customers, cleaned_inventory, and cleaned_clickstream already have the prefix.

cleaned_product_master = prod_df
cleaned_store_master = store_df
cleaned_sales_transactions=cleaned_sales
cleaned_clickstream_events=cleaned_clickstream
cleaned_inventory_data=cleaned_inventory
cleaned_customer_data=cleaned_customers

print("DataFrames have been renamed/aliased with 'cleaned_' prefix:")




In [0]:
# A professional way to standardize ALL column names in one go
def standardize_columns(df):
    for col_name in df.columns:
        # Convert to lowercase and replace any spaces/dots with underscores
        standard_name = col_name.lower().replace(" ", "_").replace(".", "_")
        df = df.withColumnRenamed(col_name, standard_name)
    return df

# Apply to all your DataFrames
cleaned_sales_transactions = standardize_columns(cleaned_sales_transactions)
cleaned_product_master = standardize_columns(cleaned_product_master)
cleaned_customer_data = standardize_columns(cleaned_customer_data)
cleaned_store_master = standardize_columns(cleaned_store_master)
cleaned_inventory_data = standardize_columns(cleaned_inventory_data)
cleaned_clickstream_events = standardize_columns(cleaned_clickstream_events)

In [0]:
print("--- Displaying all Cleaned DataFrames ---")

# Create a dictionary with the final cleaned and standardized DataFrames
cleaned_dataframes_final = {
    "cleaned_sales_transactions": cleaned_sales_transactions,
    "cleaned_product_master": cleaned_product_master,
    "cleaned_store_master": cleaned_store_master,
    "cleaned_customer_data": cleaned_customer_data,
    "cleaned_inventory_data": cleaned_inventory_data,
    "cleaned_clickstream_events": cleaned_clickstream_events
}

for name, df in cleaned_dataframes_final.items():
    print(f"\n{'='*20} {name.upper()} {'='*20}")
    print(f"Schema for {name}:")
    df.printSchema()
    print(f"Sample data for {name}:")
    df.show(5, truncate=False)


### Storing Cleaned DataFrames as CSV Files

I will now save each of the cleaned DataFrames into CSV format. Each DataFrame will be saved to its own subdirectory within `./cleaned_csv_data/` to keep the files organized.

In [0]:
# csv_output_directory = "./cleaned_csv_data/"
#
# # Save each cleaned DataFrame to a CSV file
# for name, df in cleaned_dataframes_final.items():
#     output_path = f"{csv_output_directory}{name}.csv"
#     print(f"Saving {name} to {output_path}...")
#     # Use .write.mode("overwrite").csv() to save as CSV
#     # coalescing to 1 partition for a single CSV file, good for smaller datasets
#     df.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_path)
#     print(f"Successfully saved {name}.")
#
# print("\nAll cleaned DataFrames saved to CSV files.")

### Storing Cleaned DataFrames as Parquet Files

I will now save each of the cleaned DataFrames into Parquet format. Parquet files are columnar storage files that are highly efficient for analytical queries and work well within the Spark ecosystem. Each DataFrame will be saved to its own directory with the corresponding cleaned name (e.g., `cleaned_sales_transactions.parquet`).

In [0]:
# output_directory = "./cleaned_data/"
#
# # Save each cleaned DataFrame to a Parquet file
# for name, df in cleaned_dataframes_final.items():
#     output_path = f"{output_directory}{name}.parquet"
#     print(f"Saving {name} to {output_path}...")
#     # Use .write.mode("overwrite") to ensure previous runs don't cause issues
#     df.write.mode("overwrite").parquet(output_path)
#     print(f"Successfully saved {name}.")
#
# print("\nAll cleaned DataFrames saved to Parquet files.")

In [0]:
print("--- Displaying all Cleaned DataFrames ---")

# Create a dictionary with the final cleaned and standardized DataFrames
cleaned_dataframes_final = {
    "cleaned_sales_transactions": cleaned_sales_transactions,
    "cleaned_product_master": cleaned_product_master,
    "cleaned_store_master": cleaned_store_master,
    "cleaned_customer_data": cleaned_customer_data,
    "cleaned_inventory_data": cleaned_inventory_data,
    "cleaned_clickstream_events": cleaned_clickstream_events
}

for name, df in cleaned_dataframes_final.items():
    print(f"\n{'='*20} {name.upper()} {'='*20}")
    print(f"Schema for {name}:")
    df.printSchema()
    print(f"Sample data for {name}:")
    df.show(5, truncate=False)

In [0]:
print("--- Distinct Genders in Cleaned Customer Data ---")
cleaned_customer_data.select("gender").distinct().show()

### Checking for `reorder_level` set to 0 in `cleaned_inventory_data`

This identifies products that will never trigger an automatic reorder alert because their reorder threshold is set to zero.

In [0]:
from pyspark.sql.functions import col

print("--- Checking for `reorder_level` = 0 ---")

zero_reorder_level_products = cleaned_inventory_data.filter(col("reorder_level") == 0)
zero_reorder_count = zero_reorder_level_products.count()

if zero_reorder_count > 0:
    print(f"⚠️ Found {zero_reorder_count} records where `reorder_level` is 0. These items will not trigger reorder alerts.")
    print("Sample of products with `reorder_level` = 0:")
    zero_reorder_level_products.show(truncate=False)
else:
    print("✅ No products found with `reorder_level` set to 0.")

### Checking for Stale Inventory Data (`last_updated` older than 2 years)

This identifies inventory records where the `last_updated` timestamp is older than two years from the current date, indicating potentially outdated or 'useless' inventory figures.

In [0]:
from pyspark.sql.functions import col, current_date, date_sub

print("--- Checking for Stale Inventory Data ---")

# Define the threshold for stale data (2 years ago from the current date)
stale_date_threshold = date_sub(current_date(), 365 * 2) # Approximately 2 years

stale_inventory_records = cleaned_inventory_data.filter(col("last_updated") < stale_date_threshold)
stale_inventory_count = stale_inventory_records.count()

if stale_inventory_count > 0:
    print(f"⚠️ Found {stale_inventory_count} inventory records where `last_updated` is older than 2 years.")
    print(f"Stale data threshold: {stale_date_threshold.cast('string')}. Sample of stale records:")
    stale_inventory_records.show(truncate=False)
else:
    print("✅ No stale inventory records found (all `last_updated` dates are within the last 2 years).")

### Clickstream `event_type` Standardization Check

This code verifies that the `event_type` column in `cleaned_clickstream_events` contains only the allowed standardized values: 'view', 'add_to_cart', or 'purchase'. It will display any non-standard entries found.

In [0]:
from pyspark.sql.functions import col

print("\n--- Checking Clickstream `event_type` Standardization ---")

allowed_event_types = ["view", "add_to_cart", "purchase"]

non_standard_event_types = cleaned_clickstream_events.filter(~col("event_type").isin(allowed_event_types)).distinct()

non_standard_count = non_standard_event_types.count()

if non_standard_count > 0:
    print(f"⚠️ Found {non_standard_count} records with non-standard 'event_type' values:")
    non_standard_event_types.select("event_type").show(truncate=False)
else:
    print("✅ All 'event_type' values are standardized ('view', 'add_to_cart', or 'purchase').")

### Clickstream `platform` Consistency Check

This code displays all unique values from the `platform` column in `cleaned_clickstream_events`. This allows for a visual inspection to identify any potential inconsistencies, beyond just casing, in the platform data (e.g., 'Android' vs 'Mobile App').

In [0]:
print("--- Distinct `platform` values in cleaned_clickstream_events ---")
cleaned_clickstream_events.select("platform").distinct().show(truncate=False)

### Clickstream `event_timestamp` Time Travel Check (Purchase Before View)

This code identifies inconsistencies where a 'purchase' event is recorded with an `event_timestamp` earlier than a 'view' event for the same `customer_id` and `product_id`. This scenario indicates a logical impossibility or a data quality issue.

In [0]:
from pyspark.sql.functions import col, min, when

print("--- Checking for 'Time Travel' in `event_timestamp` (Purchase Before View) ---")

time_travel_events = cleaned_clickstream_events.groupBy("customer_id", "product_id") \
    .agg(
        min(when(col("event_type") == "view", col("event_timestamp"))).alias("first_view_time"),
        min(when(col("event_type") == "purchase", col("event_timestamp"))).alias("first_purchase_time")
    ) \
    .filter(
        (col("first_purchase_time").isNotNull()) &
        (col("first_view_time").isNotNull()) &
        (col("first_purchase_time") < col("first_view_time"))
    )

time_travel_count = time_travel_events.count()

if time_travel_count > 0:
    print(f"⚠️ Found {time_travel_count} instances where a 'purchase' event occurred before a 'view' event for the same customer and product:")
    time_travel_events.show(truncate=False)
else:
    print("✅ No 'time travel' events (purchase before view) found for the same customer and product.")

In [0]:
from pyspark.sql.functions import col, lit, when

# Identify only the customer_id and product_id pairs that are anomalous
# This variable should still be available from previous execution
# anomalous_customer_product_pairs = time_travel_events.select("customer_id", "product_id").distinct()

# Assuming time_travel_events is still in scope, otherwise re-run the cell 'a59a7ce5' to get it.
# If time_travel_events is not available, we would need to regenerate it.
# Given the context, it should still be there.
anomalous_customer_product_pairs = time_travel_events.select("customer_id", "product_id").distinct()

# Add the new flag column to cleaned_clickstream_events
# We'll perform a left join and set the flag based on whether a match is found
cleaned_clickstream_events = cleaned_clickstream_events.alias("cce").join(
    anomalous_customer_product_pairs.alias("pairs"),
    on=["customer_id", "product_id"],
    how="left_outer"
).withColumn(
    "is_time_travel_anomaly",
    # If there's a match in 'pairs', it's an anomaly, otherwise it's not.
    # We check if 'pairs.customer_id' is null after the left join to determine this.
    when(col("pairs.customer_id").isNotNull(), True).otherwise(False)
).drop(col("pairs.customer_id")).drop(col("pairs.product_id")) # Drop the joined columns to clean up

print("--- Sample of cleaned_clickstream_events with 'is_time_travel_anomaly' flag ---")
cleaned_clickstream_events.select(
    "customer_id",
    "product_id",
    "event_type",
    "event_timestamp",
    "is_time_travel_anomaly"
).show(20, truncate=False)

# Verify the count of flagged anomalies
flagged_anomaly_count = cleaned_clickstream_events.filter(col("is_time_travel_anomaly") == True).count()
print(f"Total records flagged with 'is_time_travel_anomaly' = True: {flagged_anomaly_count}")



"During the Clickstream audit, I identified 2,517 'Time Travel' anomalies where purchase events preceded view events. I diagnosed this as a potential Clock Skew or Direct API interaction. To handle this, I implemented a Metadata Flagging strategy. I preserved the records for financial auditing but excluded them from the 'Marketing Funnel' analysis to ensure our Conversion Lead Time metrics remained accurate and untainted by system synchronization issues."

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

print("--- Standardizing City-State Mappings ---")

# 1. Determine the most frequent state for each city
window_spec = Window.partitionBy("city").orderBy(desc("count"))

canonical_states = cleaned_store_master.groupBy("city", "state") \
                                      .count() \
                                      .withColumn("row_num", row_number().over(window_spec)) \
                                      .filter(col("row_num") == 1) \
                                      .select(col("city"), col("state").alias("canonical_state"))

# 2. Update cleaned_store_master with the canonical state
cleaned_store_master = cleaned_store_master.join(
    canonical_states,
    on="city",
    how="left"
).withColumn("state", col("canonical_state")) \
 .drop("canonical_state")

print("City-state mappings standardized. Displaying sample of updated store_master:")
cleaned_store_master.show(5, truncate=False)


In [0]:
from pyspark.sql.functions import when, col

print("--- Simplifying City Names ---")

cleaned_store_master = cleaned_store_master.withColumn(
    "city",
    when(col("city") == "Chandigarh City", "Chandigarh")
    .when(col("city") == "New Delhi", "Delhi")
    .otherwise(col("city"))
)

print("--- Unique Cities After Simplification ---")
cleaned_store_master.select("city").distinct().show(truncate=False)

In [0]:
from pyspark.sql.functions import when, col

print("--- Standardizing City Names and Applying City-to-State Mappings ---")

# Step 1: Standardize city names (Delhi -> New Delhi, Chandigarh -> Chandigarh City)
cleaned_store_master = cleaned_store_master.withColumn(
    "city",
    when(col("city") == "Delhi", "New Delhi")
    .when(col("city") == "Chandigarh", "Chandigarh City")
    .otherwise(col("city"))
)

# Step 2: Apply state mappings based on the (now updated) city names
cleaned_store_master = cleaned_store_master.withColumn(
    "state",
    when(col("city") == "Mumbai", "Maharashtra")
    .when(col("city") == "Jaipur", "Rajasthan")
    .when(col("city") == "Bangalore", "Karnataka")
    .when(col("city") == "Pune", "Maharashtra")
    .when(col("city") == "New Delhi", "Delhi")
    .when(col("city") == "Chandigarh City", "Chandigarh")
    .when(col("city") == "Hyderabad", "Telangana")
    .otherwise(col("state")) # Keep existing state for other cities if any
)

print("--- Unique States After All Mappings ---")
cleaned_store_master.select("state").distinct().show(truncate=False)

print("--- Unique Cities After All Mappings ---")
cleaned_store_master.select("city").distinct().show(truncate=False)

In [0]:
print("--- List of Unique States in the Dataset ---")
cleaned_store_master.select("state").distinct().show(truncate=False)

### Re-checking General City/State Mapping Logic in `cleaned_store_master` (After Cleaning)

This code re-identifies inconsistencies in the `cleaned_store_master` DataFrame where a single city is mapped to more than one unique state, confirming if the issue persists after general cleaning steps.

In [0]:
from pyspark.sql.functions import countDistinct, col

print("--- Re-checking General City/State Mapping in cleaned_store_master ---")

# Group by city and count distinct states associated with each city
conflicting_city_state_mappings_cleaned = cleaned_store_master.groupBy("city") \
                                           .agg(countDistinct("state").alias("distinct_states")) \
                                           .filter("distinct_states > 1")

inconsistency_count_cleaned = conflicting_city_state_mappings_cleaned.count()

if inconsistency_count_cleaned > 0:
    print(f"⚠️ Found {inconsistency_count_cleaned} cities mapped to more than one state in cleaned_store_master:")
    conflicting_city_state_mappings_cleaned.show(truncate=False)
    print("\n--- Sample of records for these inconsistent cities in cleaned_store_master ---")
    # Display actual records for these conflicting cities for further inspection
    cleaned_store_master.join(conflicting_city_state_mappings_cleaned, "city").orderBy("city").show(truncate=False)
else:
    print("✅ No cities found that are mapped to more than one state in cleaned_store_master.")

In [0]:
print("--- Displaying all Cleaned delta tables ---")

# Create a dictionary with the final cleaned and standardized DataFrames
cleaned_dataframes_final = {
    "cleaned_sales_transactions": cleaned_sales_transactions,
    "cleaned_product_master": cleaned_product_master,
    "cleaned_store_master": cleaned_store_master,
    "cleaned_customer_data": cleaned_customer_data,
    "cleaned_inventory_data": cleaned_inventory_data,
    "cleaned_clickstream_events": cleaned_clickstream_events
}

for name, df in cleaned_dataframes_final.items():
    print(f"\n{'='*20} {name.upper()} {'='*20}")
    print(f"Schema for {name}:")
    df.printSchema()
    print(f"Sample data for {name}:")
    df.show(5, truncate=False)

In [0]:
# Save each cleaned DataFrame as a Delta table in capstone_catalog.silver schema
for name, df in cleaned_dataframes_final.items():
    table_name = f"capstone_catalog.silver.{name}"
    print(f"Saving {name} as Delta table: {table_name} ...")
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)
    print(f"Successfully saved {name} as Delta table.")

print("\nAll cleaned DataFrames saved as Delta tables in capstone_catalog.silver schema.")

GOLD LAYER

1. The Master Join (The "Single Version of Truth")
First, we create the comprehensive table that powers most of your reports.

In [0]:
from pyspark.sql import functions as F

# Create the Master Gold Table
gold_sales = cleaned_sales_transactions \
    .join(cleaned_product_master, "product_id", "left") \
    .join(cleaned_customer_data, "customer_id", "left") \
    .join(cleaned_store_master, "store_id", "left")

# Add a Profit Column
gold_sales = gold_sales.withColumn(
    "profit", 
    F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
)

In [0]:
# 1. First, create your lookup table
avg_selling_price_per_category = prod_df.groupBy("category") \
    .agg(F.avg("selling_price").alias("avg_category_selling_price"))

# 2. Join with Sales (Safe Version)
sales_with_prices = cleaned_sales.join(
    prod_df.select("product_id", "selling_price", "category"), 
    on="product_id",
    how="left"
).withColumnRenamed("selling_price", "master_selling_price")

# 3. Join with Average Price - USE STRING JOIN TO AVOID DUPLICATES
# By using on="category" (the string), Spark automatically keeps only ONE column named 'category'
sales_with_customer_info = sales_with_customer_info.join(
    avg_selling_price_per_category,
    on="category", 
    how="left"
)

# 4. Master Gold Join (If you are getting the error here)
# Make sure cleaned_sales_transactions DOES NOT already have category before this join
gold_sales = cleaned_sales_transactions.drop("category") \
    .join(cleaned_product_master, "product_id", "left") \
    .join(cleaned_customer_data, "customer_id", "left") \
    .join(cleaned_store_master, "store_id", "left") \
    .withColumn(
        "profit",
        F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
    )

from pyspark.sql import functions as F

# =====================================================
# MASTER GOLD TABLE — "Single Version of Truth"
# Joins: Sales + Products + Customers + Stores
# Handles duplicate columns (category, loyalty_status, city)
# =====================================================

gold_sales = cleaned_sales_transactions \
    .drop("category", "loyalty_status") \
    .join(cleaned_product_master, "product_id", "left") \
    .join(
        cleaned_customer_data.withColumnRenamed("city", "customer_city"),
        "customer_id", "left"
    ) \
    .join(cleaned_store_master, "store_id", "left") \
    .withColumn(
        "profit",
        F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
    )

print("=== MASTER GOLD TABLE SCHEMA ===")
gold_sales.printSchema()
print(f"\nTotal Records: {gold_sales.count():,}")
print("\nSample — 'A Gold customer in Mumbai Mall bought an Apple iPhone via Mobile App':")
gold_sales.select(
    "transaction_id", "order_date", "channel", "loyalty_status",
    "city", "store_type", "brand", "product_name", "category",
    "total_amount", "profit"
).show(5, truncate=False)

In [0]:
from pyspark.sql import functions as F

# 1. Join with Product Master
# We drop 'category' from the sales side BEFORE joining so we only keep the one from Product Master
sales_with_products = cleaned_sales_transactions.drop("category") \
    .join(cleaned_product_master, on="product_id", how="left")

# 2. Complete the Master Join
# Using the string format on="column_id" automatically merges duplicate join keys
gold_sales = sales_with_products \
    .join(cleaned_customer_data, on="customer_id", how="left") \
    .join(cleaned_store_master, on="store_id", how="left")

# 3. Add the 'profit' column (solves your previous error too)
gold_sales = gold_sales.withColumn(
    "profit", 
    F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
)

print("Check: Do we have duplicate categories?")
print([c for c in gold_sales.columns if c == "category"]) # Should print ['category'] once

In [0]:
from pyspark.sql import functions as F

# 1. Prepare Customer and Store data by renaming ambiguous columns
# This prevents 'city' and 'state' from appearing twice
customers_prepped = cleaned_customer_data \
    .withColumnRenamed("city", "customer_city") \
    .withColumnRenamed("state", "customer_state")

stores_prepped = cleaned_store_master \
    .withColumnRenamed("city", "store_city") \
    .withColumnRenamed("state", "store_state")

# 2. Re-create gold_sales with a clean join
# We drop 'category' from transactions to use the one from Product Master
gold_sales = cleaned_sales_transactions.drop("category") \
    .join(cleaned_product_master, on="product_id", how="left") \
    .join(customers_prepped, on="customer_id", how="left") \
    .join(stores_prepped, on="store_id", how="left")

# 3. Add the profit column
gold_sales = gold_sales.withColumn(
    "profit", 
    F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
)

print("Gold Sales created. New city columns: 'customer_city' and 'store_city'")

In [0]:
# A. Top Performing Categories
category_performance = gold_sales.groupBy("category") \
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("profit"), 2).alias("total_profit"),
        F.count("transaction_id").alias("transaction_count"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value")
    ).orderBy(F.col("total_profit").desc())

print("=== TOP CATEGORIES BY PROFIT ===")
category_performance.show(truncate=False)



In [0]:
# B. Store Revenue by Store City (Updated Column Name)
city_store_performance = gold_sales.groupBy("store_city", "store_state", "store_type") \
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("profit"), 2).alias("total_profit"),
        F.countDistinct("transaction_id").alias("total_transactions")
    ).orderBy(F.col("total_revenue").desc())

print("=== REVENUE BY STORE CITY & TYPE ===")
city_store_performance.show(30, truncate=False)

### 3. Conversion Rate Analysis

Calculates the **Marketing Funnel**: View → Add-to-Cart → Purchase conversion rates from clickstream data. Excludes time-travel anomalies for accurate funnel metrics. Also breaks down conversion by platform.

In [0]:
from pyspark.sql import functions as F

# =====================================================
# CONVERSION RATE: (Total Purchases / Total Views)
# Excludes time-travel anomalies for accurate funnel
# =====================================================

clean_clickstream = cleaned_clickstream_events.filter(F.col("is_time_travel_anomaly") == False)

# Event distribution
print("=== EVENT TYPE DISTRIBUTION ===")
clean_clickstream.groupBy("event_type").count().orderBy("count").show()

total_views = clean_clickstream.filter(F.col("event_type") == "view").count()
total_add_to_cart = clean_clickstream.filter(F.col("event_type") == "add_to_cart").count()
total_purchases = clean_clickstream.filter(F.col("event_type") == "purchase").count()

overall_conversion = round((total_purchases / total_views) * 100, 2) if total_views > 0 else 0
view_to_cart = round((total_add_to_cart / total_views) * 100, 2) if total_views > 0 else 0
cart_to_purchase = round((total_purchases / total_add_to_cart) * 100, 2) if total_add_to_cart > 0 else 0

print("=== CONVERSION FUNNEL ===")
print(f"Total Views:         {total_views:,}")
print(f"Total Add-to-Cart:   {total_add_to_cart:,}")
print(f"Total Purchases:     {total_purchases:,}")
print(f"\nView → Purchase Rate:     {overall_conversion}%")
print(f"View → Add-to-Cart Rate:  {view_to_cart}%")
print(f"Cart → Purchase Rate:     {cart_to_purchase}%")

# Conversion Rate by Platform
conversion_by_platform = clean_clickstream.groupBy("platform") \
    .agg(
        F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("views"),
        F.sum(F.when(F.col("event_type") == "add_to_cart", 1).otherwise(0)).alias("add_to_carts"),
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchases")
    ).withColumn("conversion_rate_pct", F.round((F.col("purchases") / F.col("views")) * 100, 2))

print("\n=== CONVERSION RATE BY PLATFORM ===")
conversion_by_platform.orderBy(F.col("conversion_rate_pct").desc()).show(truncate=False)

### 4. Stock-out Risk Identification

Identifies products where `stock_on_hand < reorder_level` — these are at immediate risk of running out of stock. Enriched with product and store details for actionable insights.

In [0]:
from pyspark.sql import functions as F

# =====================================================
# STOCK-OUT RISK: stock_on_hand < reorder_level
# =====================================================

stockout_risk = cleaned_inventory_data \
    .filter(F.col("stock_on_hand") < F.col("reorder_level")) \
    .join(cleaned_product_master.select("product_id", "product_name", "brand", "category"), "product_id", "left") \
    .join(cleaned_store_master.select("store_id", "store_name", "city"), "store_id", "left") \
    .select(
        "product_id", "product_name", "brand", "category",
        "store_id", "store_name", "city",
        "stock_on_hand", "reorder_level",
        (F.col("reorder_level") - F.col("stock_on_hand")).alias("units_below_threshold")
    ).orderBy(F.col("units_below_threshold").desc())

stockout_count = stockout_risk.count()
print(f"=== STOCK-OUT RISK: {stockout_count} product-store combinations at risk ===")
stockout_risk.show(20, truncate=False)

# Summary by category
stockout_by_category = stockout_risk.groupBy("category") \
    .agg(
        F.count("*").alias("at_risk_count"),
        F.sum("units_below_threshold").alias("total_units_deficit")
    ).orderBy(F.col("at_risk_count").desc())

print("=== STOCK-OUT RISK BY CATEGORY ===")
stockout_by_category.show(truncate=False)

# Summary by city
stockout_by_city = stockout_risk.groupBy("city") \
    .agg(F.count("*").alias("at_risk_count")) \
    .orderBy(F.col("at_risk_count").desc())

print("=== STOCK-OUT RISK BY CITY ===")
stockout_by_city.show(truncate=False)

### 5. Churn Risk Analysis

Identifies customers who **haven't purchased in the last 90 days** but **were active on the app** (clickstream). Uses data-relative dates (max dates from actual data) as reference points.

In [0]:
from pyspark.sql import functions as F

# =====================================================
# CHURN RISK: No purchase in 90 days but active on app
# Uses data-relative dates as reference
# =====================================================

# Determine reference dates from data
reference_purchase_date = cleaned_sales_transactions.agg(F.max("order_date")).collect()[0][0]
reference_activity_date = cleaned_clickstream_events.agg(F.max("event_timestamp")).collect()[0][0]

print(f"Reference Date (latest purchase in data): {reference_purchase_date}")
print(f"Reference Date (latest app activity):     {reference_activity_date}")

# Last purchase date per customer
last_purchase = cleaned_sales_transactions \
    .groupBy("customer_id") \
    .agg(F.max("order_date").alias("last_purchase_date"))

# Last clickstream activity per customer (excluding anomalies)
last_activity = cleaned_clickstream_events \
    .filter(F.col("is_time_travel_anomaly") == False) \
    .groupBy("customer_id") \
    .agg(F.max("event_timestamp").alias("last_app_activity"))

# Churn risk: no purchase in 90 days but active on app within 90 days
churn_risk = last_purchase.join(last_activity, "customer_id", "inner") \
    .withColumn("days_since_purchase", F.datediff(F.lit(reference_purchase_date), F.col("last_purchase_date"))) \
    .withColumn("days_since_app_activity", F.datediff(F.lit(reference_purchase_date), F.col("last_app_activity").cast("date"))) \
    .filter(
        (F.col("days_since_purchase") > 90) &
        (F.col("days_since_app_activity") <= 90)
    ) \
    .join(cleaned_customer_data, "customer_id", "left") \
    .select(
        "customer_id", "gender", "age", "loyalty_status", "city",
        "last_purchase_date", "last_app_activity",
        "days_since_purchase", "days_since_app_activity"
    ).orderBy(F.col("days_since_purchase").desc())

churn_count = churn_risk.count()
print(f"\n=== CHURN RISK: {churn_count} customers at risk ===")
print("(Haven't purchased in >90 days but active on app within 90 days)")
churn_risk.show(20, truncate=False)

# Churn risk by loyalty status
churn_by_loyalty = churn_risk.groupBy("loyalty_status") \
    .agg(F.count("*").alias("churn_risk_count")) \
    .orderBy(F.col("churn_risk_count").desc())

print("=== CHURN RISK BY LOYALTY STATUS ===")
churn_by_loyalty.show(truncate=False)

### 6. Customer 360 View — "Out of the Box" Analysis

Joins **Customer_Data** with **Clickstream** to see how many "Views" it takes for a customer to make a "Purchase", segmented by **loyalty tier** (Gold, Silver, Bronze). Also shows overall engagement metrics per tier.

In [0]:
from pyspark.sql import functions as F

# =====================================================
# CUSTOMER 360: Views Before Purchase by Loyalty Tier
# =====================================================

clean_clickstream = cleaned_clickstream_events.filter(F.col("is_time_travel_anomaly") == False)

# Join clickstream with customer data for loyalty status
clickstream_with_loyalty = clean_clickstream.join(
    cleaned_customer_data.select("customer_id", "loyalty_status", "gender", "age"),
    "customer_id", "left"
)

# Count views and purchases per customer per product
customer_product_funnel = clickstream_with_loyalty.groupBy("customer_id", "product_id", "loyalty_status") \
    .agg(
        F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("view_count"),
        F.sum(F.when(F.col("event_type") == "add_to_cart", 1).otherwise(0)).alias("cart_count"),
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchase_count")
    )

# Filter to only journeys that ended in a purchase
purchasers = customer_product_funnel.filter(F.col("purchase_count") > 0)

# Average views before purchase by loyalty tier
views_before_purchase = purchasers.groupBy("loyalty_status") \
    .agg(
        F.round(F.avg("view_count"), 2).alias("avg_views_before_purchase"),
        F.round(F.avg("cart_count"), 2).alias("avg_cart_adds_before_purchase"),
        F.count("*").alias("total_purchase_journeys")
    ).orderBy("loyalty_status")

print("=== CUSTOMER 360: Views Before Purchase by Loyalty Tier ===")
views_before_purchase.show(truncate=False)

# Overall customer engagement summary by loyalty tier
customer_engagement = clickstream_with_loyalty.groupBy("customer_id", "loyalty_status") \
    .agg(
        F.count("*").alias("total_events"),
        F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("total_views"),
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("total_purchases"),
        F.countDistinct("product_id").alias("unique_products_interacted")
    )

engagement_by_loyalty = customer_engagement.groupBy("loyalty_status") \
    .agg(
        F.round(F.avg("total_events"), 2).alias("avg_events_per_customer"),
        F.round(F.avg("total_views"), 2).alias("avg_views_per_customer"),
        F.round(F.avg("total_purchases"), 2).alias("avg_purchases_per_customer"),
        F.round(F.avg("unique_products_interacted"), 2).alias("avg_products_browsed"),
        F.count("*").alias("total_customers")
    ).orderBy("loyalty_status")

print("=== CUSTOMER ENGAGEMENT BY LOYALTY TIER ===")
engagement_by_loyalty.show(truncate=False)

### 7. Cross-Channel Analysis — "Out of the Box"

Answers: *Did a customer who browsed a phone on the app (Clickstream) eventually buy it in a physical store (Sales)?* Identifies customers whose **online browsing** (Mobile/App platform) led to **in-store purchases** (channel = 'Store').

In [0]:
from pyspark.sql import functions as F

# =====================================================
# CROSS-CHANNEL: Browsed on App → Bought in Store
# =====================================================

clean_clickstream = cleaned_clickstream_events.filter(F.col("is_time_travel_anomaly") == False)

# Customers who viewed products on Mobile/App platform
app_browsers = clean_clickstream \
    .filter(
        (F.col("event_type") == "view") &
        (F.col("platform").isin("Mobile", "App", "mobile", "app"))
    ) \
    .select("customer_id", "product_id").distinct()

# Customers who purchased in physical store
store_purchasers = cleaned_sales_transactions \
    .filter(F.col("channel") == "Store") \
    .select("customer_id", "product_id").distinct()

# Cross-channel: browsed on app AND bought same product in store
cross_channel = app_browsers.join(store_purchasers, ["customer_id", "product_id"], "inner") \
    .join(cleaned_customer_data, "customer_id", "left") \
    .join(cleaned_product_master.select("product_id", "product_name", "category", "brand"), "product_id", "left")

cross_channel_count = cross_channel.select("customer_id", "product_id").distinct().count()
unique_cross_customers = cross_channel.select("customer_id").distinct().count()

print(f"=== CROSS-CHANNEL ANALYSIS ===")
print(f"Total cross-channel instances (App Browse → Store Buy): {cross_channel_count:,}")
print(f"Unique customers with cross-channel behavior: {unique_cross_customers:,}")
cross_channel.show(20, truncate=False)

# By category
cross_by_category = cross_channel.groupBy("category") \
    .agg(F.countDistinct("customer_id").alias("unique_customers")) \
    .orderBy(F.col("unique_customers").desc())

print("=== CROSS-CHANNEL BY CATEGORY ===")
cross_by_category.show(truncate=False)

# By loyalty tier
cross_by_loyalty = cross_channel.groupBy("loyalty_status") \
    .agg(F.countDistinct("customer_id").alias("unique_customers")) \
    .orderBy(F.col("unique_customers").desc())

print("=== CROSS-CHANNEL BY LOYALTY TIER ===")
cross_by_loyalty.show(truncate=False)

### 8. Top Brands Analysis

Which brand (Apple, Samsung, etc.) is the **top seller**? Analyzes brand performance by revenue, profit, and units sold. Also shows the top 3 brands per category.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Prepare Tables to avoid "Ambiguous" errors
# Rename city/state so we know if it's the Customer's location or the Store's location
cust_df = cleaned_customer_data.withColumnRenamed("city", "cust_city").withColumnRenamed("state", "cust_state")
str_df = cleaned_store_master.withColumnRenamed("city", "store_city").withColumnRenamed("state", "store_state")

# 2. Perform the Master Join
# We drop 'category' from sales to use the one from the Product Master
gold_sales = cleaned_sales_transactions.drop("category") \
    .join(cleaned_product_master, on="product_id", how="left") \
    .join(cust_df, on="customer_id", how="left") \
    .join(str_df, on="store_id", how="left")

# 3. Calculate Profit (Ensuring all columns exist)
# Formula: Total Amount - (Quantity * Cost)
gold_sales = gold_sales.withColumn(
    "profit", 
    F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
)

# Verify the columns exist now
print("Verified Columns:", [c for c in gold_sales.columns if c in ["profit", "store_city", "brand"]])

In [0]:
# =====================================================
# TOP BRANDS BY REVENUE
# =====================================================
brand_performance = gold_sales.groupBy("brand") \
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("profit"), 2).alias("total_profit"),
        F.sum(F.abs(F.col("quantity"))).alias("total_units_sold"),
        F.countDistinct("transaction_id").alias("total_transactions"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value")
    ).orderBy(F.col("total_revenue").desc())

print("=== TOP BRANDS BY REVENUE ===")
brand_performance.show(truncate=False)

# =====================================================
# TOP 3 BRANDS PER CATEGORY (Window Function)
# =====================================================
# 1. First, get the revenue per brand per category
brand_revenue_df = gold_sales.groupBy("category", "brand") \
    .agg(F.round(F.sum("total_amount"), 2).alias("category_revenue"))

# 2. Define the window to rank brands within each category
window_spec = Window.partitionBy("category").orderBy(F.col("category_revenue").desc())

# 3. Apply the rank and filter
brand_by_category = brand_revenue_df \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") <= 3) \
    .orderBy("category", "rank")

print("=== TOP 3 BRANDS PER CATEGORY ===")
brand_by_category.show(50, truncate=False)

### 9. Total Revenue by City — Aggregated Gold Table

Consolidated city-level performance metrics including revenue, profit, store count, transactions, and unique customers.

In [0]:
from pyspark.sql import functions as F

# 1. Rename columns in dimension tables BEFORE the join
cust_df_clean = cleaned_customer_data \
    .withColumnRenamed("city", "cust_city") \
    .withColumnRenamed("state", "cust_state")

store_df_clean = cleaned_store_master \
    .withColumnRenamed("city", "store_city") \
    .withColumnRenamed("state", "store_state")

# 2. Re-create gold_sales
# We drop 'category' from sales to use the one from Product Master
gold_sales = cleaned_sales_transactions.drop("category") \
    .join(cleaned_product_master, on="product_id", how="left") \
    .join(cust_df_clean, on="customer_id", how="left") \
    .join(store_df_clean, on="store_id", how="left")

# 3. Add the 'profit' column (This solves your unresolved column error)
gold_sales = gold_sales.withColumn(
    "profit", 
    F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
)

print("Success: gold_sales is now clean and has a 'profit' column.")

In [0]:
# =====================================================
# TOTAL REVENUE BY CITY (Aggregated Gold Table)
# =====================================================

total_revenue_by_city = gold_sales.groupBy("store_city", "store_state") \
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("profit"), 2).alias("total_profit"),
        F.countDistinct("store_id").alias("num_stores"),
        F.countDistinct("transaction_id").alias("total_transactions"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value")
    ).orderBy(F.col("total_revenue").desc())

print("=== TOTAL REVENUE BY CITY ===")
total_revenue_by_city.show(truncate=False)

Insight 1: Which Category is the "Money Maker"?
This tells the manager where to invest more marketing budget.

In [0]:
category_analysis = gold_sales.groupBy("category") \
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.sum("profit").alias("total_profit"),
        F.count("transaction_id").alias("transaction_count")
    ).orderBy(F.col("total_profit").desc())

category_analysis.show()

Insight 2: Does Loyalty actually drive bigger Sales?
This proves if the "Gold" member program is working.

In [0]:
from pyspark.sql import functions as F

# 1. Prepare Dimension Tables to avoid overlaps
# We rename 'loyalty_status' and 'city' to make them unique
cust_df_final = cleaned_customer_data \
    .withColumnRenamed("loyalty_status", "cust_loyalty") \
    .withColumnRenamed("city", "customer_city") \
    .withColumnRenamed("state", "customer_state")

store_df_final = cleaned_store_master \
    .withColumnRenamed("city", "store_city") \
    .withColumnRenamed("state", "store_state")

# 2. Re-create the Master Gold Table
# We drop 'category' from sales to use the official one from Product Master
gold_sales = cleaned_sales_transactions.drop("category") \
    .join(cleaned_product_master, on="product_id", how="left") \
    .join(cust_df_final, on="customer_id", how="left") \
    .join(store_df_final, on="store_id", how="left")

# 3. Add the 'profit' column
gold_sales = gold_sales.withColumn(
    "profit", 
    F.round(F.col("total_amount") - (F.abs(F.col("quantity")) * F.col("cost_price")), 2)
)

print("Gold Table Rebuilt. Use 'cust_loyalty' instead of 'loyalty_status' now.")

In [0]:
# Analysis: Does Loyalty drive bigger Sales?
loyalty_analysis = gold_sales.groupBy("cust_loyalty") \
    .agg(
        F.round(F.avg("total_amount"), 2).alias("avg_order_value"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("profit"), 2).alias("total_profit")
    ).orderBy(F.col("avg_order_value").desc())

print("=== LOYALTY STATUS PERFORMANCE ===")
loyalty_analysis.show(truncate=False)

**Insight 3: The "Return" Red Flag
Which products are being returned the most? (Using your transaction_type flag).**

In [0]:
return_analysis = gold_sales.filter(F.col("transaction_type") == "RETURN") \
    .groupBy("product_name", "category") \
    .agg(F.count("transaction_id").alias("return_count")) \
    .orderBy(F.col("return_count").desc())

print("Top 5 Most Returned Products:")
return_analysis.show(5)


### 10. Save All Gold Tables to `capstone_catalog.gold`

Persists all analytical gold tables as Delta tables for dashboarding and reporting.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- 1. AOV and Channel Analysis ---
aov_by_channel = gold_sales.groupBy("channel").agg(
    F.round(F.avg("total_amount"), 2).alias("avg_order_value"),
    F.round(F.sum("total_amount"), 2).alias("total_revenue")
)

# --- 2. Conversion and Platform Analysis ---
# Calculation: (Purchases / Views) * 100
total_views = cleaned_clickstream.filter(F.col("event_type") == "view").count()
conversion_by_platform = cleaned_clickstream.groupBy("platform").agg(
    (F.count(F.when(F.col("event_type") == "purchase", 1)) / 
     F.count(F.when(F.col("event_type") == "view", 1)) * 100).alias("conversion_rate")
)

# --- 3. Stock and Risk Analysis ---
stockout_risk = cleaned_inventory_data.filter(F.col("stock_on_hand") < F.col("reorder_level"))
stockout_by_category = stockout_risk.join(cleaned_product_master, "product_id").groupBy("category").count()

# --- 4. Churn and Loyalty Analysis ---
# Customers in Clickstream but not in Sales
churn_risk = cleaned_clickstream.select("customer_id").distinct().join(
    cleaned_sales_transactions.select("customer_id").distinct(), "customer_id", "left_anti"
)
churn_by_loyalty = churn_risk.join(cleaned_customer_data, "customer_id").groupBy("loyalty_status").count()

# --- 5. Engagement and Cross-Channel ---
engagement_by_loyalty = cleaned_clickstream.join(cleaned_customer_data, "customer_id").groupBy("loyalty_status", "event_type").count()
views_before_purchase = gold_sales.groupBy("customer_id").agg(F.count("transaction_id").alias("purchase_count")) # Placeholder for Customer 360
cross_by_category = gold_sales.groupBy("category", "channel").agg(F.sum("total_amount").alias("revenue"))

In [0]:
gold_tables = {
    "gold_master_sales": gold_sales,
    "gold_category_performance": category_performance,
    "gold_city_store_performance": city_store_performance,
    "gold_aov_by_channel": aov_by_channel,
    "gold_conversion_by_platform": conversion_by_platform,
    "gold_stockout_risk": stockout_risk,
    "gold_stockout_by_category": stockout_by_category,
    "gold_churn_risk": churn_risk,
    "gold_churn_by_loyalty": churn_by_loyalty,
    "gold_views_before_purchase": views_before_purchase,
    "gold_customer_engagement_by_loyalty": engagement_by_loyalty,
    "gold_cross_channel_by_category": cross_by_category,
    "gold_brand_performance": brand_performance,
    "gold_brand_by_category": brand_by_category,
    "gold_total_revenue_by_city": total_revenue_by_city
}

for name, df in gold_tables.items():
    table_name = f"capstone_catalog.gold.{name}"
    print(f"Saving {name} → {table_name} ...")
    # Saving as Delta Tables for ACID compliance
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    print(f"✅ Saved {name}")

A single table that tells you: "A Gold customer in a Mumbai Mall bought an Apple iPhone via the Mobile App."


In [0]:
from pyspark.sql import functions as F

# 1. Rename columns in dimension tables to avoid "Ambiguous" errors
# This ensures we know exactly which city/status we are looking at
cust_df_clean = cleaned_customer_data \
    .withColumnRenamed("city", "customer_city") \
    .withColumnRenamed("loyalty_status", "cust_loyalty")

store_df_clean = cleaned_store_master \
    .withColumnRenamed("city", "store_city") \
    .withColumnRenamed("store_name", "mall_name")

# 2. Create the Master Gold Table
# We join all 4 tables using their IDs
gold_master_sales = cleaned_sales_transactions.drop("category") \
    .join(cleaned_product_master, on="product_id", how="left") \
    .join(cust_df_clean, on="customer_id", how="left") \
    .join(store_df_clean, on="store_id", how="left")

# 3. Add a "Human Readable Insight" column
# This creates the exact sentence you asked for
gold_master_sales = gold_master_sales.withColumn(
    "transaction_story",
    F.concat(
        F.lit("A "), F.col("cust_loyalty"), F.lit(" customer in "), 
        F.col("mall_name"), F.lit(" bought an "), 
        F.col("brand"), F.lit(" "), F.col("product_name"), 
        F.lit(" via the "), F.col("channel")
    )
)

# 4. Show the result
gold_master_sales.select("transaction_id", "transaction_story").show(5, truncate=False)

In [0]:
# Filtering for the specific Mumbai/Mobile App example
mumbai_insight = gold_master_sales.filter(
    (F.col("store_city") == "Mumbai") & 
    (F.col("channel") == "Mobile App") & 
    (F.col("cust_loyalty") == "Gold")
)

mumbai_insight.select("transaction_story").show(truncate=False)